# ZEW 4.0 — Analysis notebook
**The Financial Lifespan of Zombie Companies** · Integrating machine learning and survival analysis

Runs the full ZEW 4.0 framework on the data collected by the pipeline notebook (`ZEW_Data_Pipeline.ipynb`):

| Part | Spec section | Output |
|---|---|---|
| 0–1 | — | install packages, write the code (16 modules) |
| 2 | — | settings: real data or demo data |
| 3 | §02 Layer 0 | analysis panel + attrition funnel |
| 4 | §03 Layer 1 | the Gate — zombie / subsidised zombie / at risk |
| 5 | §04 Layer 2 | severity 0–100 and root-cause pillars |
| 6 | §05 Layer 3 | states, transitions, trajectories |
| 7 | §06 Layer 4 | six published definitions + Altman, Beneish, Piotroski |
| 8 | §09 | fuzzy variant |
| 9 | §10 | validation V1–V6 |
| 10 | Objective 4 | survival: Kaplan–Meier, log-rank, Cox, death clock |
| 11 | Objectives 2–3 | machine learning: 24-month probability of default |
| 12 | §01 research question | forensic: do zombies manipulate earnings? |
| 13 | Objective 5 | risk dashboard + Excel / HTML report |

Put this notebook **in the same folder as the pipeline notebook**, so it finds `zew_data/`.
No real data yet? Set `USE_DEMO = True` in Part 2 to run everything on synthetic data first.

## Part 0 — Install packages (run once)

In [ ]:
%pip install -q pandas numpy scipy matplotlib scikit-learn statsmodels lifelines xgboost imbalanced-learn openpyxl

## Part 1 — Write the analysis code
Each cell saves one module of the `zew_analysis` package next to this notebook. Run them all; no edits needed.

In [ ]:
import os
os.makedirs("zew_analysis", exist_ok=True)
os.makedirs("tests", exist_ok=True)
print("ready:", os.getcwd())

**`zew_analysis/__init__.py`**

In [ ]:
%%writefile zew_analysis/__init__.py
"""ZEW 4.0 analysis: gate, severity, states, benchmarks, validation, survival, ML, forensic, charts."""
__version__ = "4.0.0"

import warnings as _w
_w.filterwarnings("ignore", category=FutureWarning)
_w.filterwarnings("ignore", category=RuntimeWarning)


**`zew_analysis/config.py`**

In [ ]:
%%writefile zew_analysis/config.py
"""
ZEW 4.0 analysis — every threshold and design constant in one place.

Values marked (spec) come straight from the ZEW 4.0 Framework Specification; values
marked (impl) are implementation choices the spec leaves open — each one is written
into the methodology table produced by report.py so nothing is hidden.
"""
import os

DATA_DIR = os.environ.get("ZEW_OUT", "zew_data")            # output folder of the data pipeline
RESULTS_DIR = os.environ.get("ZEW_RESULTS", "zew_results")  # tables written here
FIG_DIR = os.path.join(RESULTS_DIR, "figures")              # charts written here
SEED = 2026

# ───────────────────────────── Layer 0 — universe
FY_MIN, FY_MAX = 2015, 2026
MIN_USABLE_YEARS = 5            # (spec 0.3)
MIN_SECTOR_FIRMS = 8            # (spec 0.4) smaller buckets collapse to "Other"
WINSOR = (0.01, 0.99)           # (spec 0.5) for distributions / severity only
FILING_LAG_MONTHS = 2           # (impl) SEBI LODR: audited annual results within 60 days

# ───────────────────────────── Layer 1 — gate
ICR_CUT = 1.0                   # (spec) G1, G2 break-even
PERSIST_YEARS = 3               # (spec) AAM18; BH20 variant = 2
AGE_MIN = 10                    # (spec) AAM18; sensitivity 5 / 10 / 15
Q_PCTL = 0.50                   # (spec) BH20 sector median; sensitivity 0.25
Q_YEARS_NEEDED = 2              # (spec) 2 of last 3 years
CFO_NEG_YEARS = 2               # (spec) G6
EFF_RATE_PCTL = 0.25            # (spec) G7; sensitivity 0.10
VETO_ICR = 3.0                  # (spec) G9 — anchor to CRISIL/ICRA/CARE criteria; grid 2–4
VETO_ND_EBITDA = 2.0            # (spec) G9; grid 1.5–3
AGE_UNKNOWN_POLICY = "pass"     # (impl) age unknown & listing < 10 y ago: "pass" (flagged) or "fail"

# ───────────────────────────── Layer 2 — severity
Z_CLIP = 4.0                    # (spec)
MAD_K = 1.4826                  # (spec)
PILLAR_WEIGHTS = {"P1": 0.35, "P2": 0.25, "P3": 0.20, "P4": 0.15, "P5": 0.05}   # (spec)
OVERRIDE = 0.75                 # (spec) ZEW = max(linear, 0.75 × max pillar)
MIN_PILLARS = 3                 # (spec) fewer → Data_Insufficient
MIN_VARS_IN_PILLAR = 0.5        # (impl) pillar computable if ≥ half its variables exist
# variable: (pillar, risk direction +1 = higher is riskier, absolute anchor or None)
SEVERITY_VARS = {
    "ICR":               ("P1", -1, 1.0),
    "CashICR":           ("P1", -1, 1.0),
    "DSCR":              ("P1", -1, 1.0),
    "ND_EBITDA":         ("P1", +1, 4.0),     # (impl) anchor 4× = sub-investment-grade band
    "FCF_Margin_3y":     ("P2", -1, 0.0),
    "CFO_to_EBITDA":     ("P2", -1, None),
    "Sloan_Accruals":    ("P2", +1, None),
    "DSO_Index":         ("P2", +1, None),
    "ND_Assets":         ("P3", +1, None),
    "ST_Debt_Share":     ("P3", +1, None),
    "Debt_EBITDA_Div":   ("P3", +1, 0.0),
    "EffRate_Gap":       ("P3", -1, None),    # below-sector rate = forbearance → risk
    "ROCE_minus_WACC":   ("P4", -1, 0.0),
    "EBITDA_Margin_Slope": ("P4", -1, 0.0),
    "RE_TA":             ("P4", -1, 0.0),
    "Merton_DD":         ("P5", -1, None),
    "Q_Demeaned":        ("P5", -1, None),
    "Excess_Ret_3y":     ("P5", -1, 0.0),
}

# ───────────────────────────── Layer 3 — states
BAND_HEALTHY, BAND_WATCH, BAND_ATRISK = 40, 55, 70        # (spec) severity bands
WATCH_EXIT, ATRISK_EXIT = 45, 55                          # (spec)
RECOVERY_ICR = 1.5                                        # (spec) ICR > 1.5 ∧ FCF > 0, 2 yrs
TRAJ_WINDOW = 3                                           # (spec) rolling OLS window
STATE_ORDER = ["Healthy", "Watch", "At Risk", "Zombie", "Subsidised Zombie",
               "Recovered", "Data Insufficient"]

# ───────────────────────────── Layer 4 — benchmarks
DEFINITIONS = ["ZEW4", "AAM18", "BH20", "CHK08", "SKSW17", "ACEE20", "SST17"]
ALTMAN_ZONES = (4.35, 5.85)     # Altman EM Z″ (with 3.25 constant): distress < 4.35 < grey < 5.85 < safe
BENEISH_CUT = -1.78             # Beneish (1999) 8-variable cut-off; −2.22 reported as robustness
SKSW_DSC = 0.05                 # Storz et al.: debt-servicing capacity < 5 %
SST_LEV_PCTL = 0.75             # (impl) Schivardi "high leverage" = sector-year p75

# ───────────────────────────── validation
KNOWN_GOOD = ["TCS", "INFY", "HINDUNILVR", "NESTLEIND", "ASIANPAINT", "ITC", "MARUTI",
              "BAJAJ-AUTO", "HEROMOTOCO", "EICHERMOT", "TITAN", "PIDILITIND", "BRITANNIA",
              "DABUR", "COLPAL", "MARICO", "WIPRO", "DIVISLAB", "PAGEIND", "HAVELLS",
              "BOSCHLTD", "COALINDIA", "POWERGRID", "ULTRACEMCO", "BERGEPAINT"]   # V2 — edit if needed
BOOTSTRAP_REPS = 200            # V6 flip-rate replications
NOISE_SD = 0.05                 # V6 ±5 % multiplicative measurement noise
NULL_FIRMS = 751                # V1 synthetic healthy universe size (spec example)

# ───────────────────────────── survival / ML
EVENT_COL = "Event_Flag_Substantive"   # headline death definition (fallback: Event_Flag)
PD_HORIZON_MONTHS = 24          # (impl) label = IBC admission within 24 m of filing date
DATA_CUTOFF = "2026-09-30"      # last date events were collected
TRAIN_LAST_FY, TEST_FIRST_FY = 2021, 2022

# ───────────────────────────── chart palette (validated reference palette)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
STATUS = {"good": "#0ca30c", "warning": "#fab219", "serious": "#ec835a", "critical": "#d03b3b"}
STATE_COLORS = {"Healthy": STATUS["good"], "Watch": STATUS["warning"], "At Risk": STATUS["serious"],
                "Zombie": STATUS["critical"], "Subsidised Zombie": "#8f1f1f",
                "Recovered": "#2a78d6", "Data Insufficient": "#b4b2ab"}
SEQ = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
DIVERGING = ("#2a78d6", "#f0efec", "#e34948")
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#8a8984", "#e6e5e1", "#fcfcfb"


**`zew_analysis/data.py`**

In [ ]:
%%writefile zew_analysis/data.py
"""
Layer 0 — load the pipeline output and build the analysis panel.

Rules implemented (spec §02, Layer 0):
  0.1 point-in-time universe with dead firms retained   (pipeline step 1, kept as is)
  0.2 exclude financial firms                           (Is_Financial / bank-format pages)
  0.3 >= 5 consecutive usable years                     (else Data_Insufficient, counted)
  0.4 sector attached; buckets < 8 firms → "Other"
  0.5 winsorise 1/99 within sector-year — severity layer & distributions only
"""
import os

import numpy as np
import pandas as pd

from . import config as C

KEY = ["ISIN_Root", "FY"]
USABLE_FIELDS = ["Revenue", "EBIT", "Interest", "CFO", "Total_Assets", "Total_Debt"]


def _p(name):
    return os.path.join(C.DATA_DIR, name)


def read(name, **kw):
    p = _p(name)
    if not os.path.exists(p):
        return pd.DataFrame()
    return pd.read_csv(p, low_memory=False, **kw)


def _bool(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1", "1.0", "yes"])


def load_panel(verbose=True):
    abt = read("09_abt_firm_year.csv")
    if abt.empty:
        raise SystemExit(f"{_p('09_abt_firm_year.csv')} not found — run the data pipeline (step 9) first, "
                         "or use zew_analysis.synthetic.make_demo_data() to try the analysis.")
    abt["FY"] = pd.to_numeric(abt.FY, errors="coerce").astype("Int64")
    abt = abt.dropna(subset=["ISIN_Root", "FY"]).copy()
    abt["FY"] = abt.FY.astype(int)
    abt = abt[(abt.FY >= C.FY_MIN) & (abt.FY <= C.FY_MAX)]
    abt = abt.sort_values(KEY).drop_duplicates(KEY, keep="last").reset_index(drop=True)

    funnel = [("firm-years in ABT", len(abt), abt.ISIN_Root.nunique())]

    # 0.2 financial firms out
    fin = _bool(abt.get("Is_Financial", pd.Series(False, index=abt.index))) | \
        _bool(abt.get("Bank_Format", pd.Series(False, index=abt.index)))
    abt = abt[~fin].copy()
    funnel.append(("after excluding financial firms (0.2)", len(abt), abt.ISIN_Root.nunique()))

    # firm-years after the death event are not at risk
    if "Post_Event" in abt:
        abt = abt[~_bool(abt.Post_Event)].copy()

    # numeric coercion
    skip = {"ISIN_Root", "Screener_Key", "Company", "ISIN", "NSE_Symbol", "Industry", "Index_Today",
            "Cohort", "CIN", "Inc_Source", "Statement_Basis", "Shares_Source", "Date_Source",
            "Censor_Reason", "Exit_Type", "source_note", "CIN_NIC"}
    for c in abt.columns:
        if c not in skip and abt[c].dtype == object:
            conv = pd.to_numeric(abt[c], errors="coerce")
            if conv.notna().sum() >= 0.5 * abt[c].notna().sum():
                abt[c] = conv
    for c in ("Event_Date", "Censor_Date", "Incorporation_Date", "NSE_Listing_Date", "PIT_First", "PIT_Last"):
        if c in abt:
            abt[c] = pd.to_datetime(abt[c], errors="coerce")

    # 0.3 usable years
    abt["Usable"] = abt[[c for c in USABLE_FIELDS if c in abt]].notna().all(axis=1)
    nuse = abt.groupby("ISIN_Root").Usable.sum()
    ok = nuse[nuse >= C.MIN_USABLE_YEARS].index
    abt["Data_Insufficient_Firm"] = ~abt.ISIN_Root.isin(ok)
    funnel.append((f"firms with >= {C.MIN_USABLE_YEARS} usable years (0.3)",
                   int(abt[~abt.Data_Insufficient_Firm].shape[0]), len(ok)))

    # 0.4 sector buckets
    sec = abt.get("Industry", pd.Series("Unknown", index=abt.index)).fillna("Unknown").astype(str)
    firm_sec = sec.groupby(abt.ISIN_Root).agg(lambda s: s.mode().iat[0] if len(s.mode()) else "Unknown")
    counts = firm_sec.value_counts()
    small = set(counts[counts < C.MIN_SECTOR_FIRMS].index)
    abt["Sector"] = abt.ISIN_Root.map(firm_sec).where(lambda s: ~s.isin(small), "Other")

    # dates
    abt["FY_End"] = pd.to_datetime(abt.FY.astype(str) + "-03-31")
    if "Period_End_Max" in abt:
        pe = pd.to_datetime(abt.Period_End_Max, errors="coerce")
        abt["FY_End"] = pe.fillna(abt.FY_End)
    abt["Filing_Date"] = abt.FY_End + pd.DateOffset(months=C.FILING_LAG_MONTHS)

    ev_col = C.EVENT_COL if C.EVENT_COL in abt else "Event_Flag"
    abt["Death"] = pd.to_numeric(abt.get(ev_col, 0), errors="coerce").fillna(0).astype(int)
    abt["Death_Any"] = pd.to_numeric(abt.get("Event_Flag", 0), errors="coerce").fillna(0).astype(int)
    funnel.append(("IBC deaths (headline definition) among usable firms",
                   int(abt[~abt.Data_Insufficient_Firm].drop_duplicates("ISIN_Root").Death.sum()), None))
    if verbose:
        for s, n, f in funnel:
            print(f"  {s:60s} {n:>7,}" + (f"  firm-years / {f:,} firms" if f is not None else ""))
    abt.attrs["funnel"] = funnel
    return abt.reset_index(drop=True)


def winsorise(s, by):
    lo, hi = C.WINSOR
    q = s.groupby(by).transform(lambda x: x.quantile(lo)), s.groupby(by).transform(lambda x: x.quantile(hi))
    return s.clip(q[0], q[1])


def load_prices():
    """Weekly firm prices (04c) and index levels (04b) for beta / excess return."""
    px = read("04c_prices_weekly.csv.gz")
    ix = read("04b_index_prices.csv")
    for d in (px, ix):
        if len(d):
            d["Date"] = pd.to_datetime(d.Date, errors="coerce")
    return px, ix


**`zew_analysis/ratios.py`**

In [ ]:
%%writefile zew_analysis/ratios.py
"""
All derived variables (spec §03 gate G1–G9, §04 pillars P1–P5, India diagnostics,
Altman / Beneish / Piotroski inputs, ML velocity features). Raw inputs come from the
pipeline's analytical base table; nothing here is scraped.

Lags are taken by fiscal year, not by row, so a gap in a firm's history never
silently pairs non-adjacent years.
"""
import warnings

import numpy as np
import pandas as pd

from . import config as C

warnings.filterwarnings("ignore", category=RuntimeWarning)

BIG = 99.0                     # stand-in for "no debt service due" in coverage ratios


def lag(df, col, k=1):
    """Value of `col` k fiscal years earlier for the same firm (NaN if that year is missing)."""
    s = df.set_index(["ISIN_Root", "FY"])[col]
    idx = pd.MultiIndex.from_arrays([df.ISIN_Root, df.FY - k])
    return pd.Series(s.reindex(idx).values, index=df.index)


def roll(df, col, n, fn="sum", min_periods=None):
    """Rolling statistic over the last n fiscal years (t-n+1..t), year-aligned."""
    vals = [lag(df, col, k) if k else df[col] for k in range(n)]
    m = pd.concat(vals, axis=1)
    mp = n if min_periods is None else min_periods
    ok = m.notna().sum(axis=1) >= mp
    out = getattr(m, fn)(axis=1)
    return out.where(ok)


def count_true(df, col, n):
    m = pd.concat([lag(df, col, k) if k else df[col] for k in range(n)], axis=1)
    return m.sum(axis=1, min_count=1).where(m.notna().sum(axis=1) == n)


def safe_div(a, b):
    a, b = pd.to_numeric(a, errors="coerce"), pd.to_numeric(b, errors="coerce")
    return (a / b).where(b.abs() > 1e-9)


def slope3(df, col):
    """OLS slope of col on time over t-2..t (needs all 3)."""
    y0, y1, y2 = lag(df, col, 2), lag(df, col, 1), df[col]
    return ((y2 - y0) / 2.0).where(y0.notna() & y1.notna() & y2.notna())


def sector_stat(df, col, q=0.5):
    return df.groupby(["Sector", "FY"])[col].transform(lambda x: x.quantile(q) if x.notna().sum() >= 3 else np.nan)


# ════════════════════════════════════════════════════════════ beta from weekly prices
def betas(df, px, ix, window_weeks=104):
    """Blume-adjusted 2-year weekly beta vs Nifty 500 at each FY end. Fallback 1.0."""
    out = pd.Series(np.nan, index=df.index)
    if px is None or ix is None or px.empty or ix.empty or "Screener_Key" not in df:
        return out
    n5 = ix[ix.Index == "NIFTY500"].set_index("Date").Price.sort_index()
    n5w = n5.resample("W-FRI").last().pct_change()
    for key, g in px.groupby("Screener_Key"):
        r = g.set_index("Date").Price.sort_index().resample("W-FRI").last().pct_change()
        j = pd.concat([r.rename("r"), n5w.rename("m")], axis=1).dropna()
        if len(j) < 30:
            continue
        rows = df.index[df.Screener_Key == key]
        for i in rows:
            end = df.at[i, "FY_End"]
            w = j[(j.index <= end) & (j.index > end - pd.Timedelta(weeks=window_weeks))]
            if len(w) >= 30 and w.m.var() > 0:
                b = np.cov(w.r, w.m)[0, 1] / w.m.var()
                out.at[i] = 0.67 * b + 0.33
    return out


# ════════════════════════════════════════════════════════════ main
def compute(df, px=None, ix=None):
    d = df.copy()
    g = lambda c: pd.to_numeric(d[c], errors="coerce") if c in d else pd.Series(np.nan, index=d.index)

    TA, TD, INT = g("Total_Assets"), g("Total_Debt"), g("Interest")
    EBIT, EBITDA, CFO, REV = g("EBIT"), g("EBITDA"), g("CFO"), g("Revenue")
    d["Has_Debt"] = (TD.fillna(0) > 0) & (INT.fillna(0) > 0)

    # ── G1 / G2 coverage
    d["ICR"] = np.where(INT > 0, EBIT / INT, BIG)
    d["CashICR"] = np.where(INT > 0, CFO / INT, BIG)
    d.loc[EBIT.isna() | INT.isna(), "ICR"] = np.nan
    d.loc[CFO.isna() | INT.isna(), "CashICR"] = np.nan
    d["ICR_lt1"] = (d.ICR < C.ICR_CUT).astype(float).where(d.ICR.notna())
    d["CashICR_lt1"] = (d.CashICR < C.ICR_CUT).astype(float).where(d.CashICR.notna())

    # ── G3 age (+ lower bound from listing date)
    inc = g("Inc_Year")
    d["Firm_Age"] = d.FY - inc
    lst = pd.to_datetime(d.get("NSE_Listing_Date"), errors="coerce") if "NSE_Listing_Date" in d else pd.Series(pd.NaT, index=d.index)
    d["Listing_Age"] = d.FY - lst.dt.year
    d["Age_Source"] = np.where(d.Firm_Age.notna(), "incorporation",
                               np.where(d.Listing_Age.notna(), "listing_lower_bound", "unknown"))

    # ── G4 Tobin's Q
    mcap = g("MCap_Cr")
    d["Q"] = safe_div(mcap + TD.fillna(0), TA).where(mcap.notna())
    d["Q_SectorMed"] = sector_stat(d, "Q", 0.5)
    d["Q_SectorP25"] = sector_stat(d, "Q", 0.25)
    d["Q_Demeaned"] = d.Q - d.Q_SectorMed

    # ── G5 / G6 cash
    d["FCF"] = g("FCF") if "FCF" in d else CFO - g("Capex")
    d["FCF_3y"] = roll(d, "FCF", 3)
    d["Rev_3y"] = roll(d.assign(_r=REV), "_r", 3)
    d["FCF_Margin_3y"] = safe_div(d.FCF_3y, d.Rev_3y)
    d["CFO_neg"] = (CFO < 0).astype(float).where(CFO.notna())
    d["CFO_neg_3y"] = count_true(d, "CFO_neg", 3)

    # ── G7 effective rate
    d["Avg_Debt"] = (TD + lag(d, "Total_Debt")).div(2).fillna(TD)
    d["EffRate"] = safe_div(INT, d.Avg_Debt).where(d.Avg_Debt > 0)
    d["EffRate"] = d.EffRate.where(d.EffRate.between(0, 1))          # drop accounting artefacts
    d["EffRate_SectorMed"] = sector_stat(d, "EffRate", 0.5)
    d["EffRate_P25"] = sector_stat(d, "EffRate", C.EFF_RATE_PCTL)
    d["EffRate_P10"] = sector_stat(d, "EffRate", 0.10)
    d["EffRate_Gap"] = d.EffRate - d.EffRate_SectorMed

    # ── G8 debt–EBITDA divergence (3-year log spread)
    TD3, E3 = lag(d, "Total_Debt", 3), lag(d, "EBITDA", 3)
    both = (TD > 0) & (TD3 > 0) & (EBITDA > 0) & (E3 > 0)
    div = np.log(TD / TD3) - np.log(EBITDA / E3)
    d["Debt_EBITDA_Div"] = div.where(both)
    turned = (EBITDA <= 0) & (E3 > 0) & (TD >= TD3)                 # EBITDA collapsed, debt held
    d.loc[turned, "Debt_EBITDA_Div"] = 2.0
    d["Debt_Not_Reduced_EBITDA_Fell"] = ((TD >= TD3) & (EBITDA < E3)).astype(float).where(TD3.notna() & E3.notna())

    # ── G9 healthy-veto inputs
    ND = g("Net_Debt") if "Net_Debt" in d else TD - g("Cash_Equivalents")
    d["Net_Debt"] = ND
    d["ND_EBITDA"] = np.where(EBITDA > 0, ND / EBITDA, np.where(ND > 0, BIG, 0.0))
    d.loc[EBITDA.isna() | ND.isna(), "ND_EBITDA"] = np.nan
    d["RE_TA"] = safe_div(g("Reserves"), TA)

    # ── P1 extras
    d["DSCR"] = safe_div(CFO, INT.fillna(0) + g("ST_Borrowings").fillna(0)).where(
        (INT.fillna(0) + g("ST_Borrowings").fillna(0)) > 0)
    d.loc[d.DSCR.isna() & CFO.notna() & ~d.Has_Debt, "DSCR"] = BIG

    # ── P2 cash generation & earnings quality
    d["CFO_to_EBITDA"] = safe_div(CFO, EBITDA).where(EBITDA > 0)
    avgTA = (TA + lag(d, "Total_Assets")).div(2).fillna(TA)
    d["Avg_TA"] = avgTA
    d["Sloan_Accruals"] = safe_div(g("Net_Profit") - CFO, avgTA)
    d["AR_Rev"] = safe_div(g("Trade_Receivables"), REV)
    d["DSO_Index"] = safe_div(d.AR_Rev, lag(d, "AR_Rev"))

    # ── P3 capital structure fragility
    d["ND_Assets"] = safe_div(ND, TA)
    d["ST_Debt_Share"] = safe_div(g("ST_Borrowings"), TD).where(TD > 0).clip(0, 1)

    # ── P4 economic profitability
    tax = (g("Tax_Pct") / 100).where(lambda t: t.between(0, 0.45))
    stat = g("statutory_tax_rate")
    stat = stat.where(stat < 1, stat / 100).fillna(pd.Series(np.where(d.FY >= 2020, 0.2517, 0.3461), index=d.index))
    d["Tax_Rate"] = tax.fillna(stat)
    CL = g("Current_Liabilities_approx")
    cap_emp = (TA - CL).where(lambda x: x > 0)
    d["ROCE"] = safe_div(EBIT * (1 - d.Tax_Rate), cap_emp)
    beta = betas(d, px, ix) if px is not None else pd.Series(np.nan, index=d.index)
    d["Beta"] = beta.clip(0.3, 2.5).fillna(1.0)
    d["Beta_Source"] = np.where(beta.notna(), "weekly_2y", "default_1.0")
    rf = (g("gsec10y") / 100)
    d["Rf"] = rf.fillna(0.07)
    erp = g("India_Total_ERP")
    d["ERP"] = erp.where(erp < 1, erp / 100).fillna(0.08)
    d["Ke"] = d.Rf + d.Beta * d.ERP
    kd = d.EffRate.clip(0.04, 0.25).fillna(d.Rf + 0.03)
    E = mcap.where(mcap > 0, g("Total_Equity").where(g("Total_Equity") > 0))
    V = E.fillna(0) + TD.fillna(0)
    wE = safe_div(E.fillna(0), V).fillna(1.0)
    d["WACC"] = wE * d.Ke + (1 - wE) * kd * (1 - d.Tax_Rate)
    d["ROCE_minus_WACC"] = d.ROCE - d.WACC
    d["EBITDA_Margin"] = safe_div(EBITDA, REV).where(REV > 0).clip(-2, 1)
    d["EBITDA_Margin_Slope"] = slope3(d, "EBITDA_Margin")

    # ── P5 market-implied risk — naive Merton DD (Bharath & Shumway 2008)
    sE = g("Equity_Vol_Ann")
    F = (g("ST_Borrowings").fillna(0) + 0.5 * g("LT_Borrowings").fillna(0))
    F = F.where(F > 0, TD.where(TD > 0))
    r = (g("tbill3m") / 100).fillna(d.Rf)
    mu = g("Ret_1y").clip(-0.9, 1.0).fillna(r)
    sD = 0.05 + 0.25 * sE
    Vn = mcap + F
    sV = (mcap / Vn) * sE + (F / Vn) * sD
    d["Merton_DD"] = ((np.log(Vn / F) + (mu - 0.5 * sV ** 2)) / sV).where(
        mcap.notna() & F.notna() & sE.notna() & (sV > 0))
    d.loc[mcap.notna() & sE.notna() & (TD.fillna(0) <= 0), "Merton_DD"] = 10.0
    d["Merton_PD"] = pd.Series(_norm_cdf(-d.Merton_DD), index=d.index)
    p, n5 = g("Price_FY_End"), g("NIFTY500_Level_FY_End")
    d["_p"], d["_n"] = p, n5
    d["Excess_Ret_3y"] = safe_div(p, lag(d, "_p", 3)) - safe_div(n5, lag(d, "_n", 3))

    # ── India diagnostics
    d["CL_to_NetWorth"] = safe_div(g("CL_Total_Cr"), g("Total_Equity").where(g("Total_Equity") > 0))
    d["Pledge_Pct"] = g("Pledge_Pct_of_Promoter")

    # ── Altman / Beneish / Piotroski building blocks
    d["Working_Capital"] = g("Current_Assets_approx") - CL
    d["Total_Liab_Book"] = TA - g("Total_Equity")
    d["Gross_Margin"] = safe_div(REV - g("COGS"), REV).where(REV > 0)
    d["Asset_Turnover"] = safe_div(REV, lag(d, "Total_Assets"))
    d["ROA"] = safe_div(g("Net_Profit"), lag(d, "Total_Assets"))
    d["Current_Ratio"] = safe_div(g("Current_Assets_approx"), CL)
    d["LT_Lev"] = safe_div(g("LT_Borrowings"), avgTA)
    d["DE"] = safe_div(TD, g("Total_Equity").where(g("Total_Equity") > 0))

    # ── ML velocity features (spec Objective 2, expressed annually)
    for c in ("ICR", "CashICR", "DE", "ND_Assets", "EBITDA_Margin", "Current_Ratio"):
        cl = d[c].clip(-50, 50) if c in ("ICR", "CashICR") else d[c]
        d[f"_{c}"] = cl
        d[f"d_{c}"] = cl - lag(d, f"_{c}")
        d[f"dd_{c}"] = d[f"d_{c}"] - lag(d, f"d_{c}")              # acceleration
    d["WC_Burn"] = safe_div(d.Working_Capital - lag(d, "Working_Capital"), avgTA)
    d["CFO_TA"] = safe_div(CFO, avgTA)
    d["d_CFO_TA"] = d.CFO_TA - lag(d, "CFO_TA")
    d["dlog_Debt"] = np.log(TD.where(TD > 0)) - np.log(lag(d, "Total_Debt").where(lambda x: x > 0))
    d["log_TA"] = np.log(TA.where(TA > 0))
    d["Lev"] = safe_div(TD, TA)
    d = d.drop(columns=[c for c in d.columns if c.startswith("_")])
    return d


def _norm_cdf(x):
    from scipy.stats import norm
    return norm.cdf(pd.to_numeric(x, errors="coerce"))


**`zew_analysis/gate.py`**

In [ ]:
%%writefile zew_analysis/gate.py
"""
Layer 1 — the Gate (spec §03). Binary, absolute thresholds; a healthy universe returns 0.

    Step 1  HEALTHY_VETO  (ICR>3 ∧ ND/EBITDA<2 ∧ 3y FCF>0 ∧ RE/TA>0)        → Healthy, stop
    Step 2  A = has_debt ∧ (ICR<1 ×3y ∨ CashICR<1 ×3y) ∧ age ≥ 10
    Step 3  B = Q < sector median in ≥2 of last 3 y                      (BH20)
            C = 3y cumulative FCF < 0  ∨  CFO < 0 in ≥2 of 3 y
    Step 4  D = (debt not reduced ∧ EBITDA fell)  ∨  (eff. rate < sector p25 ∧ ICR<1)
    Zombie = A ∧ (B ∨ C) ;  Subsidised Zombie = Zombie ∧ D
    At Risk = A alone, or (B ∨ C) alone ;  Not a Zombie = otherwise

`run_gate(df, **params)` takes every threshold as a parameter so the sensitivity grids
(V6) and the 2-year BH20-style persistence variant reuse the exact same code.
"""
import numpy as np
import pandas as pd

from . import config as C
from .ratios import lag


def _persist(d, flag_col, n):
    """1 if flag true in each of the last n years (all observed)."""
    m = pd.concat([lag(d, flag_col, k) if k else d[flag_col] for k in range(n)], axis=1)
    return (m.sum(axis=1) == n).where(m.notna().all(axis=1))


def run_gate(df, icr_cut=C.ICR_CUT, persist=C.PERSIST_YEARS, age_min=C.AGE_MIN,
             q_pctl=C.Q_PCTL, eff_pctl=C.EFF_RATE_PCTL, veto_icr=C.VETO_ICR,
             veto_nd=C.VETO_ND_EBITDA, use_cash_icr=True, age_policy=C.AGE_UNKNOWN_POLICY):
    d = df[["ISIN_Root", "FY", "Sector"]].copy()
    d["_icr_lt"] = (df.ICR < icr_cut).astype(float).where(df.ICR.notna())
    d["_cicr_lt"] = (df.CashICR < icr_cut).astype(float).where(df.CashICR.notna())
    icr_p = _persist(d, "_icr_lt", persist)
    cicr_p = _persist(d, "_cicr_lt", persist)

    # G3 age
    age = df.Firm_Age
    lb = df.Listing_Age
    age_ok = (age >= age_min).where(age.notna())
    age_ok = age_ok.fillna((lb >= age_min).where(lb >= age_min))
    unknown = age_ok.isna()
    age_ok = age_ok.fillna(age_policy == "pass").astype(bool)

    g1 = icr_p.fillna(0).astype(bool)
    g2 = cicr_p.fillna(0).astype(bool) if use_cash_icr else pd.Series(False, index=d.index)
    A = df.Has_Debt & (g1 | g2) & age_ok

    # G4 Q below sector percentile in >=2 of 3 years (statement fallback if Q missing)
    qcut = df.Q_SectorMed if q_pctl == 0.5 else df.groupby(["Sector", "FY"]).Q.transform(lambda x: x.quantile(q_pctl))
    d["_qlow"] = (df.Q < qcut).astype(float).where(df.Q.notna() & qcut.notna())
    m = pd.concat([lag(d, "_qlow", k) if k else d["_qlow"] for k in range(3)], axis=1)
    q_obs = m.notna().sum(axis=1)
    B = (m.sum(axis=1) >= C.Q_YEARS_NEEDED) & (q_obs >= 2)
    # fallback (spec): 3-y revenue CAGR < 0 and EBITDA-margin trend < 0
    rev3 = lag(df.assign(_r=df.Revenue), "_r", 3) if "Revenue" in df else None
    if rev3 is not None:
        cagr = (df.Revenue / rev3) ** (1 / 3) - 1
        fb = (cagr < 0) & (df.EBITDA_Margin_Slope < 0)
        B = B | ((q_obs < 2) & fb.fillna(False))
    B_fallback = (q_obs < 2)

    # G5 / G6
    C_ = (df.FCF_3y < 0).fillna(False) | (df.CFO_neg_3y >= C.CFO_NEG_YEARS).fillna(False)

    # G7 / G8 escalators
    eff_cut = df.EffRate_P25 if eff_pctl == 0.25 else (df.EffRate_P10 if eff_pctl == 0.10 else
                                                         df.groupby(["Sector", "FY"]).EffRate.transform(lambda x: x.quantile(eff_pctl)))
    g7 = ((df.EffRate < eff_cut) & (df.ICR < icr_cut)).fillna(False)
    g8 = (df.Debt_Not_Reduced_EBITDA_Fell == 1).fillna(False)
    D = g7 | g8

    # G9 veto
    veto = ((df.ICR > veto_icr) & (df.ND_EBITDA < veto_nd) & (df.FCF_3y > 0) & (df.RE_TA > 0)).fillna(False)

    BC = B | C_
    zombie = A & BC & ~veto
    label = np.select([veto, zombie & D, zombie, (A & ~BC) | (~A & BC)],
                      ["Healthy (veto)", "Subsidised Zombie", "Zombie", "At Risk"], "Not a Zombie")
    out = pd.DataFrame({
        "G1_ICR_persist": g1, "G2_CashICR_persist": g2, "G3_Age_OK": age_ok, "Age_Unknown": unknown,
        "A": A, "B": B, "B_Fallback_Used": B_fallback, "C": C_, "D": D, "G7_Subsidy": g7,
        "G8_Evergreen": g8, "Veto": veto, "Gate_Label": label,
        "Gate_Zombie": zombie, "Gate_Subsidised": zombie & D,
        # observability: need `persist` years of coverage history
        "Gate_Computable": icr_p.notna() | cicr_p.notna(),
        "Incremental_CashICR": A & ~g1,          # A fired only through cash coverage (G2)
    }, index=df.index)
    return out


def apply(df):
    g = run_gate(df)
    g2 = run_gate(df, persist=2)
    g["Gate_Zombie_2y"] = g2.Gate_Zombie
    g["Gate_Zombie_noCashICR"] = run_gate(df, use_cash_icr=False).Gate_Zombie
    return pd.concat([df, g], axis=1)


**`zew_analysis/severity.py`**

In [ ]:
%%writefile zew_analysis/severity.py
"""
Layer 2 — severity and root cause (spec §04). Never produces the zombie label.

    z      = (x − median_sector,year) / (1.4826 · MAD_sector,year),  clip ±4
    score  = 100 / (1 + exp(−z))                       50 = sector median
    blended (variables with an economic threshold):
    score  = 100 / (1 + exp(−0.5·z_rel − 0.5·z_abs)),  z_abs = (x − threshold) / (1.4826 · MAD)
    pillar = max(mean of its variable scores, 0.75 · worst variable)        (impl: non-compensatory)
    ZEW_linear = 0.35·P1 + 0.25·P2 + 0.20·P3 + 0.15·P4 + 0.05·P5   (weights renormalised over
                 computable pillars; < 3 pillars → Data Insufficient)
    ZEW_4.0    = max(ZEW_linear, 0.75 · max(P1…P5))

All z-scores are oriented so that a HIGHER score means MORE distress. Inputs are
winsorised 1/99 within sector-year first (spec 0.5).
"""
import numpy as np
import pandas as pd

from . import config as C
from .data import winsorise


def _robust_z(x, by):
    med = x.groupby(by).transform("median")
    mad = (x - med).abs().groupby(by).transform("median") * C.MAD_K
    # degenerate MAD (e.g. most firms identical) → fall back to the pooled-year MAD
    pooled = (x - x.median()).abs().median() * C.MAD_K
    mad = mad.where(mad > 1e-9, pooled if pooled > 1e-9 else np.nan)
    return (x - med) / mad, mad


def score_variables(df):
    by = [df.Sector, df.FY]
    out = pd.DataFrame(index=df.index)
    for var, (pillar, sign, anchor) in C.SEVERITY_VARS.items():
        if var not in df or df[var].notna().sum() < 10:
            continue
        x = pd.to_numeric(df[var], errors="coerce").replace([np.inf, -np.inf], np.nan)
        x = winsorise(x, by)
        z, mad = _robust_z(x, by)
        z = (sign * z).clip(-C.Z_CLIP, C.Z_CLIP)
        if anchor is not None:
            za = (sign * (x - anchor) / mad).clip(-C.Z_CLIP, C.Z_CLIP)
            zz = 0.5 * z + 0.5 * za
        else:
            zz = z
        out[f"S_{var}"] = 100.0 / (1.0 + np.exp(-zz))
    return out


def pillars(scores):
    res = pd.DataFrame(index=scores.index)
    for p in C.PILLAR_WEIGHTS:
        cols = [f"S_{v}" for v, (pp, _, _) in C.SEVERITY_VARS.items() if pp == p and f"S_{v}" in scores]
        if not cols:
            res[p] = np.nan
            continue
        m = scores[cols]
        need = max(1, int(np.ceil(C.MIN_VARS_IN_PILLAR * len(cols))))
        ok = m.notna().sum(axis=1) >= need
        res[p] = np.maximum(m.mean(axis=1), C.OVERRIDE * m.max(axis=1)).where(ok)
    return res


def aggregate(P):
    w = pd.Series(C.PILLAR_WEIGHTS)
    avail = P.notna()
    wsum = (avail * w).sum(axis=1)
    linear = (P.fillna(0) * w).sum(axis=1) / wsum.replace(0, np.nan)
    zew = np.maximum(linear, C.OVERRIDE * P.max(axis=1))
    n = avail.sum(axis=1)
    insuff = n < C.MIN_PILLARS
    return pd.DataFrame({"ZEW_Linear": linear.where(~insuff), "ZEW": zew.where(~insuff),
                         "N_Pillars": n, "Severity_Insufficient": insuff,
                         "Top_Pillar": P.idxmax(axis=1).where(~insuff)})


def equal_weight(P):
    """Robustness: equal pillar weights."""
    lin = P.mean(axis=1)
    return np.maximum(lin, C.OVERRIDE * P.max(axis=1)).where(P.notna().sum(axis=1) >= C.MIN_PILLARS)


def leave_one_out(P):
    out = {}
    w = pd.Series(C.PILLAR_WEIGHTS)
    for p in P.columns:
        Q = P.drop(columns=p)
        ww = w.drop(p)
        lin = (Q.fillna(0) * ww).sum(axis=1) / (Q.notna() * ww).sum(axis=1).replace(0, np.nan)
        out[f"ZEW_without_{p}"] = np.maximum(lin, C.OVERRIDE * Q.max(axis=1))
    return pd.DataFrame(out)


def apply(df):
    s = score_variables(df)
    P = pillars(s)
    agg = aggregate(P)
    res = pd.concat([df, s, P, agg], axis=1)
    res["ZEW_EqualWeight"] = equal_weight(P)
    return pd.concat([res, leave_one_out(P)], axis=1)


**`zew_analysis/states.py`**

In [ ]:
%%writefile zew_analysis/states.py
"""
Layer 3 — persistence state machine and trajectory (spec §05).

  State              Entry                                         Exit
  Healthy            veto, or ZEW < 40 for 2 consecutive years     ZEW ≥ 55
  Watch              ZEW ∈ [55, 70)                                ZEW < 45 for 2 consecutive years
  At Risk            ZEW ≥ 70, or gate condition A alone           ZEW < 55 for 2 consecutive years
  Zombie             gate: A ∧ (B ∨ C)                             ICR > 1.5 ∧ FCF > 0, 2 yrs → Recovered
  Subsidised Zombie  gate ∧ (G7 ∨ G8)                              as above
  Data Insufficient  < 3 computable pillars                        —

Implementation choices (impl): a firm's first classifiable year takes the band state
directly (40–55 with no history → Healthy). A Zombie stays Zombie in later years even
if the gate stops firing, until the recovery rule is met — zombie status is chronic.

Trajectory: OLS slope of ZEW on time over a rolling 3-year window with its SE;
"Deteriorating" if slope > 1.96·SE, "Improving" if slope < −1.96·SE, else "Stable".
With 3 points the SE has 1 df, so the test is deliberately conservative.
"""
import numpy as np
import pandas as pd

from . import config as C


def _firm_states(g):
    st, prev = [], None
    zew = g.ZEW.values
    icr, fcf = g.ICR.values, g.FCF.values
    zombie, subs, A, veto, insuff = (g.Gate_Zombie.values, g.Gate_Subsidised.values,
                                     g.A.values, g.Veto.values, g.Severity_Insufficient.values)
    low40 = low45 = low55 = rec = 0
    for i in range(len(g)):
        z = zew[i]
        low40 = low40 + 1 if (z == z and z < C.BAND_HEALTHY) else 0
        low45 = low45 + 1 if (z == z and z < C.WATCH_EXIT) else 0
        low55 = low55 + 1 if (z == z and z < C.ATRISK_EXIT) else 0
        rec = rec + 1 if (icr[i] == icr[i] and icr[i] > C.RECOVERY_ICR and fcf[i] == fcf[i] and fcf[i] > 0) else 0
        if zombie[i]:
            s = "Subsidised Zombie" if subs[i] else "Zombie"
        elif prev in ("Zombie", "Subsidised Zombie"):
            s = "Recovered" if rec >= 2 else prev
        elif insuff[i] and not A[i]:
            s = "Data Insufficient"
        elif veto[i]:
            s = "Healthy"
        elif A[i] or z >= C.BAND_ATRISK:
            s = "At Risk"
        elif prev == "At Risk" and low55 < 2:
            s = "At Risk"
        elif C.BAND_WATCH <= z < C.BAND_ATRISK:
            s = "Watch"
        elif prev == "Watch" and low45 < 2:
            s = "Watch"
        elif prev in (None, "Data Insufficient", "Recovered") or low40 >= 2:
            s = "Healthy" if z < C.BAND_WATCH else "Watch"
        else:
            s = prev if prev in ("Healthy",) and z < C.BAND_WATCH else ("Healthy" if z < C.BAND_WATCH else "Watch")
        st.append(s)
        prev = s
    return st


def trajectory(df):
    """Rolling 3-year OLS slope of ZEW with SE (time = 0,1,2)."""
    from .ratios import lag
    y0, y1, y2 = lag(df, "ZEW", 2), lag(df, "ZEW", 1), df.ZEW
    ok = y0.notna() & y1.notna() & y2.notna()
    slope = (y2 - y0) / 2.0
    fitted_mid = (y0 + y1 + y2) / 3.0
    resid = [y0 - (fitted_mid - slope), y1 - fitted_mid, y2 - (fitted_mid + slope)]
    sse = sum(r ** 2 for r in resid)
    se = np.sqrt(sse / 1.0) / np.sqrt(2.0)             # Sxx = 2, df = n−2 = 1
    cls = np.where(slope > 1.96 * se, "Deteriorating", np.where(slope < -1.96 * se, "Improving", "Stable"))
    return pd.DataFrame({"ZEW_Slope": slope.where(ok), "ZEW_Slope_SE": se.where(ok),
                         "Trajectory": pd.Series(cls, index=df.index).where(ok)})


def apply(df):
    d = df.sort_values(["ISIN_Root", "FY"]).copy()
    d["State"] = None
    for root, g in d.groupby("ISIN_Root", sort=False):
        d.loc[g.index, "State"] = _firm_states(g)
    d["Is_Zombie_State"] = d.State.isin(["Zombie", "Subsidised Zombie"])
    d = pd.concat([d, trajectory(d)], axis=1)
    # entry / exit events — needed by the survival layer
    prev = d.groupby("ISIN_Root").Is_Zombie_State.shift(1).fillna(False).infer_objects(copy=False).astype(bool)
    d["Zombie_Entry"] = d.Is_Zombie_State & ~prev
    d["Zombie_Exit"] = (d.State == "Recovered") & prev
    return d.sort_index()


def transition_matrix(d):
    nxt = d.sort_values(["ISIN_Root", "FY"]).groupby("ISIN_Root").State.shift(-1)
    t = pd.crosstab(d.State, nxt, normalize="index")
    order = [s for s in C.STATE_ORDER if s in t.index]
    cols = [s for s in C.STATE_ORDER if s in t.columns]
    return t.reindex(index=order, columns=cols).fillna(0)


**`zew_analysis/benchmarks.py`**

In [ ]:
%%writefile zew_analysis/benchmarks.py
"""
Layer 4 — parallel benchmarks (spec §06). None of these feed the ZEW label.

Six published zombie definitions (+ ZEW 4.0 itself = 7):
  AAM18   Adalet McGowan et al. (2018): ICR < 1 for 3 consecutive years ∧ age ≥ 10
  BH20    Banerjee & Hofmann (2018/22): ICR < 1 for 2 consecutive years ∧ Q < sector median
  CHK08   Caballero, Hoshi & Kashyap (2008): interest paid < hypothetical minimum
          R* = r_ST(t−1)·ST debt(t−1) + mean(r_LT, t−5..t−1)·LT debt(t−1)
          r_ST = prime-rate proxy (manual SBI MCLR/base rate → World Bank lending rate),
          r_LT = 10-y G-sec (fallback lending rate)
  SKSW17  Storz et al. (2017): RoA < 0 ∧ net investment < 0 ∧ CFO/debt < 5 %, 2 consecutive years
  ACEE20  Acharya et al. (2019/20): leverage > sector median ∧ ICR < sector median ∧
          interest rate paid < rate of the best-rated firms (AAA 5-y yield; fallback:
          median effective rate of the sample's top-ICR-quintile firms that year)
  SST17   Schivardi, Sette & Tabellini (2017): 3-y mean RoA (EBIT/TA) < prime rate ∧
          leverage > sector-year p75   (impl: "high leverage" cut)
Distress / manipulation / quality scores:
  Altman Z″ (emerging markets), Beneish M (8 variables), Piotroski F (9 tests)
"""
import numpy as np
import pandas as pd

from . import config as C
from .ratios import lag, safe_div


def _consec(d, flag, n):
    m = pd.concat([lag(d.assign(_f=flag.astype(float).where(flag.notna())), "_f", k) if k
                   else flag.astype(float).where(flag.notna()) for k in range(n)], axis=1)
    return (m.sum(axis=1) == n).where(m.notna().all(axis=1)).fillna(False).astype(bool)


def definitions(d):
    out = pd.DataFrame(index=d.index)
    icr_lt = (d.ICR < 1).where(d.ICR.notna())
    age_ok = ((d.Firm_Age >= 10).where(d.Firm_Age.notna())).fillna(d.Listing_Age >= 10)
    age_ok = age_ok.fillna(C.AGE_UNKNOWN_POLICY == "pass").astype(bool)
    out["ZEW4"] = d.Gate_Zombie.astype(bool)
    out["AAM18"] = _consec(d, icr_lt, 3) & d.Has_Debt & age_ok
    out["BH20"] = _consec(d, icr_lt, 2) & d.Has_Debt & (d.Q < d.Q_SectorMed).fillna(False)

    # CHK08
    prime = d.get("sbi_mclr_1y_or_base_rate", pd.Series(np.nan, index=d.index))
    prime = prime.fillna(d.get("lending_rate", pd.Series(np.nan, index=d.index)))
    prime = prime.where(prime < 1, prime / 100)
    rlt = d.get("gsec10y", pd.Series(np.nan, index=d.index))
    rlt = rlt.where(rlt < 1, rlt / 100).fillna(prime)
    d2 = d.assign(_prime=prime, _rlt=rlt)
    r_st = lag(d2, "_prime", 1)
    r_lt = pd.concat([lag(d2, "_rlt", k) for k in range(1, 6)], axis=1).mean(axis=1)
    rstar = r_st * lag(d, "ST_Borrowings", 1).fillna(0) + r_lt * lag(d, "LT_Borrowings", 1).fillna(0)
    out["CHK08"] = ((d.Interest < rstar) & (rstar > 0) & d.Has_Debt).where(rstar.notna()).fillna(False).astype(bool)

    # SKSW17
    roa = safe_div(d.EBIT, d.Total_Assets)
    net_inv = d.get("Net_Capex", d.get("Capex")) - d.Depreciation
    dsc = safe_div(d.CFO, d.Total_Debt).where(d.Total_Debt > 0)
    sk = ((roa < 0) & (net_inv < 0) & (dsc < C.SKSW_DSC)).where(roa.notna() & dsc.notna())
    out["SKSW17"] = _consec(d, sk, 2)

    # ACEE20
    lev = d.Lev
    lev_med = d.groupby(["Sector", "FY"]).Lev.transform("median")
    icr_med = d.groupby(["Sector", "FY"]).ICR.transform("median")
    aaa = d.get("aaa_5y_corporate_yield", pd.Series(np.nan, index=d.index))
    aaa = aaa.where(aaa < 1, aaa / 100)
    q80 = d.groupby("FY").ICR.transform(lambda x: x.quantile(0.8))
    best = d.EffRate.where(d.ICR >= q80).groupby(d.FY).transform("median")
    bench = aaa.fillna(best)
    out["ACEE20"] = ((lev > lev_med) & (d.ICR < icr_med) & (d.EffRate < bench) & d.Has_Debt).fillna(False)
    out["ACEE20_Benchmark_Source"] = np.where(aaa.notna(), "AAA_yield", "top_ICR_quintile_rate")

    # SST17
    roa3 = pd.concat([roa] + [lag(d.assign(_roa=roa), "_roa", k) for k in (1, 2)], axis=1).mean(axis=1)
    lev75 = d.groupby(["Sector", "FY"]).Lev.transform(lambda x: x.quantile(C.SST_LEV_PCTL))
    out["SST17"] = ((roa3 < prime) & (lev > lev75)).where(prime.notna()).fillna(False).astype(bool)
    out["Def_Computable_CHK08"] = rstar.notna()
    out["Def_Computable_SST17"] = prime.notna()
    return out


def altman(d):
    TA = d.Total_Assets
    x1 = safe_div(d.Working_Capital, TA)
    x2 = safe_div(d.Reserves, TA)
    x3 = safe_div(d.EBIT, TA)
    x4 = safe_div(d.Total_Equity, d.Total_Liab_Book.where(d.Total_Liab_Book > 0))
    z = 3.25 + 6.56 * x1 + 3.26 * x2 + 6.72 * x3 + 1.05 * x4
    lo, hi = C.ALTMAN_ZONES
    zone = np.select([z < lo, z < hi, z >= hi], ["Distress", "Grey", "Safe"], None)
    return pd.DataFrame({"Altman_Z": z, "Altman_Zone": pd.Series(zone, index=d.index).where(z.notna())})


def beneish(d):
    S, S1 = d.Revenue, lag(d, "Revenue")
    AR, AR1 = d.Trade_Receivables, lag(d, "Trade_Receivables")
    GM, GM1 = d.Gross_Margin, lag(d, "Gross_Margin")
    TA, TA1 = d.Total_Assets, lag(d, "Total_Assets")
    CA = d.Current_Assets_approx
    PPE, PPE1 = d.Net_Block, lag(d, "Net_Block")
    SEC = d.Investments.fillna(0)
    d2 = d.assign(_aq=1 - safe_div(CA + PPE + SEC, TA),
                  _dep=safe_div(d.Depreciation, d.Depreciation + PPE),
                  _sga=safe_div(d.SGA_Other_Cost, S),
                  _lv=safe_div(d.Current_Liabilities_approx + d.LT_Borrowings.fillna(0), TA))
    DSRI = safe_div(safe_div(AR, S), safe_div(AR1, S1))
    GMI = safe_div(GM1, GM)
    AQI = safe_div(d2._aq, lag(d2, "_aq"))
    SGI = safe_div(S, S1)
    DEPI = safe_div(lag(d2, "_dep"), d2._dep)
    SGAI = safe_div(d2._sga, lag(d2, "_sga"))
    LVGI = safe_div(d2._lv, lag(d2, "_lv"))
    ni = d.get("Profit_Excl_Exceptional", d.Net_Profit).fillna(d.Net_Profit)
    TATA = safe_div(ni - d.CFO, TA)
    idx = pd.DataFrame({"DSRI": DSRI, "GMI": GMI, "AQI": AQI, "SGI": SGI, "DEPI": DEPI,
                        "SGAI": SGAI, "LVGI": LVGI, "TATA": TATA})
    idx = idx.clip(lower=idx.quantile(0.01), upper=idx.quantile(0.99), axis=1)   # ratio blow-ups
    M = (-4.84 + 0.920 * idx.DSRI + 0.528 * idx.GMI + 0.404 * idx.AQI + 0.892 * idx.SGI
         + 0.115 * idx.DEPI - 0.172 * idx.SGAI + 4.679 * idx.TATA - 0.327 * idx.LVGI)
    out = idx.add_prefix("Ben_")
    out["Beneish_M"] = M
    out["Beneish_Flag"] = (M > C.BENEISH_CUT).where(M.notna())
    out["Beneish_Flag_222"] = (M > -2.22).where(M.notna())
    return out


def piotroski(d):
    roa, roa1 = d.ROA, lag(d, "ROA")
    cfo_ta = safe_div(d.CFO, lag(d, "Total_Assets"))
    sh, sh1 = d.Shares_Best if "Shares_Best" in d else d.get("Shares_from_EPS"), None
    sh1 = lag(d.assign(_s=sh), "_s")
    tests = pd.DataFrame({
        "F1_ROA_pos": roa > 0, "F2_CFO_pos": d.CFO > 0, "F3_dROA_pos": roa > roa1,
        "F4_Accrual": cfo_ta > roa, "F5_dLev_neg": d.LT_Lev < lag(d, "LT_Lev"),
        "F6_dCurrent_pos": d.Current_Ratio > lag(d, "Current_Ratio"),
        "F7_No_Issue": sh <= sh1 * 1.01, "F8_dGM_pos": d.Gross_Margin > lag(d, "Gross_Margin"),
        "F9_dTurn_pos": d.Asset_Turnover > lag(d, "Asset_Turnover")})
    avail = pd.concat([roa, d.CFO, roa1, cfo_ta, lag(d, "LT_Lev"), lag(d, "Current_Ratio"), sh1,
                       lag(d, "Gross_Margin"), lag(d, "Asset_Turnover")], axis=1).notna().sum(axis=1)
    F = tests.sum(axis=1).where(avail >= 7)
    return pd.concat([tests.astype(float), F.rename("Piotroski_F")], axis=1)


def apply(d):
    return pd.concat([d, definitions(d), altman(d), beneish(d), piotroski(d)], axis=1)


**`zew_analysis/fuzzy.py`**

In [ ]:
%%writefile zew_analysis/fuzzy.py
"""
Fuzzy variant (spec §09, after ECB WP 2743): continuous 0–1 membership instead of a crisp gate.

  μ_ICR(x) = 1 if x ≤ 0.8 ;  (1.2 − x)/0.4 if 0.8 < x < 1.2 ;  0 if x ≥ 1.2     (same for CashICR)
  μ_A      = min over the 3-year window of max(μ_ICR, μ_CashICR)  × has_debt × age_ok
  μ_B      = second-largest of the three yearly μ_Q   (fuzzy "2 of 3 years")
             μ_Q = 1 if Q ≤ 0.8·median ; 0 if Q ≥ 1.2·median ; linear between
  μ_C      = max( μ_FCF, μ_CFO )
             μ_FCF: 3-y FCF margin ≤ −2 % → 1, ≥ +2 % → 0, linear between
             μ_CFO: second-largest yearly μ of CFO/Revenue on the same ±2 % ramp
  μ_Z      = μ_A × max(μ_B, μ_C)                       fuzzy AND = product, OR = max
(impl) The ramps for Q, FCF and CFO mirror the ±20 % / break-even band the spec uses for ICR.
"""
import numpy as np
import pandas as pd

from .ratios import lag, safe_div


def ramp(x, lo, hi):
    """1 at/below lo, 0 at/above hi, linear between (lo < hi)."""
    return ((hi - x) / (hi - lo)).clip(0, 1).where(x.notna())


def _window(d, col, k=3):
    return pd.concat([lag(d, col, i) if i else d[col] for i in range(k)], axis=1)


def apply(d):
    f = d[["ISIN_Root", "FY"]].copy()
    f["mu_icr"] = np.maximum(ramp(d.ICR, 0.8, 1.2).fillna(0), ramp(d.CashICR, 0.8, 1.2).fillna(0)).where(
        d.ICR.notna() | d.CashICR.notna())
    w = _window(f, "mu_icr")
    mu_A = w.min(axis=1).where(w.notna().all(axis=1))
    mu_A = mu_A * d.Has_Debt.astype(float) * d.G3_Age_OK.astype(float)

    rel = safe_div(d.Q, d.Q_SectorMed)
    f["mu_q"] = ramp(rel, 0.8, 1.2)
    wq = _window(f, "mu_q")
    mu_B = wq.apply(lambda r: np.sort(r.dropna().values)[-2] if r.notna().sum() >= 2 else np.nan, axis=1)

    mu_fcf = ramp(d.FCF_Margin_3y, -0.02, 0.02)
    f["mu_cfo"] = ramp(safe_div(d.CFO, d.Revenue.where(d.Revenue > 0)), -0.02, 0.02)
    wc = _window(f, "mu_cfo")
    mu_cfo = wc.apply(lambda r: np.sort(r.dropna().values)[-2] if r.notna().sum() >= 2 else np.nan, axis=1)
    mu_C = pd.concat([mu_fcf, mu_cfo], axis=1).max(axis=1)

    mu_Z = mu_A * pd.concat([mu_B, mu_C], axis=1).max(axis=1)
    return pd.concat([d, pd.DataFrame({"mu_A": mu_A, "mu_B": mu_B, "mu_C": mu_C, "mu_Z": mu_Z},
                                      index=d.index)], axis=1)


**`zew_analysis/validation.py`**

In [ ]:
%%writefile zew_analysis/validation.py
"""
Validation plan (spec §10).

Without labels
  V1  null-universe simulation — every firm ICR > 4 and FCF > 0: ZEW 4.0 must return 0 zombies;
      the v2.0-style percentile rule flags ~1/3 by construction
  V2  known-good control list (config.KNOWN_GOOD) — false-positive rate must be 0
  V3  known-distressed list = IBC deaths — recall, first condition to fire, lead time
  V4  sector balance — chi-square of zombie flags × sector
  V5  cross-method agreement — pairwise Jaccard across the 7 definitions (+ "all ÷ any")
  V6  threshold-stability grids + bootstrap measurement-noise flip rates
With labels (IBC admissions)
  ROC-AUC, PR-AUC, Brier score and Youden-J threshold for every definition and score
"""
import itertools

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

from . import config as C
from . import gate as G
from . import ratios as R


# ════════════════════════════════════════════════════════════ v2.0 reconstruction
def v2_percentile_flag(d):
    """Reconstruction of the withdrawn v2.0 logic: mean cross-sectional percentile rank of
    distress ratios, top third flagged. Used only as the 'before' column in V1/V2."""
    parts = pd.DataFrame({
        "icr": -d.ICR.clip(-50, 50), "nd": d.ND_EBITDA.clip(-10, 50), "fcf": -d.FCF_Margin_3y,
        "q": -d.Q, "lev": d.Lev})
    ranks = parts.groupby(d.FY).rank(pct=True).fillna(0.5)          # v2 used fillna(50)
    composite = ranks.mean(axis=1)
    score = composite.groupby(d.FY).rank(pct=True) * 100            # final score = percentile rank
    return score > 200 / 3


# ════════════════════════════════════════════════════════════ V1
def v1_null_universe(seed=C.SEED, n=C.NULL_FIRMS):
    from .synthetic import healthy_universe
    raw = healthy_universe(n=n, seed=seed)
    d = R.compute(raw)
    g = G.run_gate(d)
    v2 = v2_percentile_flag(d)
    last = d.FY == d.FY.max()
    res = pd.DataFrame({
        "Method": ["ZEW 4.0 gate", "v2.0 percentile rule (reconstructed)"],
        "Firms": [n, n],
        "Flagged": [int(g.Gate_Zombie[last].sum()), int(v2[last].sum())],
    })
    res["Flagged_Pct"] = 100 * res.Flagged / res.Firms
    return res


# ════════════════════════════════════════════════════════════ V2
def v2_known_good(d):
    sym = d.get("NSE_Symbol", pd.Series(index=d.index, dtype=object)).astype(str).str.upper()
    sk = d.get("Screener_Key", pd.Series(index=d.index, dtype=object)).astype(str).str.upper()
    m = sym.isin(C.KNOWN_GOOD) | sk.isin(C.KNOWN_GOOD)
    g = d[m]
    if g.empty:
        return pd.DataFrame(), {}
    per = g.groupby(g.NSE_Symbol.fillna(g.Screener_Key)).agg(
        Firm_Years=("FY", "size"), ZEW_Zombie_Years=("Gate_Zombie", "sum"),
        ZEW_AtRisk_Years=("Gate_Label", lambda s: (s == "At Risk").sum()),
        v2_Flagged_Years=("v2_Flag", "sum"), Veto_Years=("Veto", "sum")).reset_index()
    summ = {"firms_found": len(per), "zew_false_positive_rate": float((per.ZEW_Zombie_Years > 0).mean()),
            "v2_false_positive_rate": float((per.v2_Flagged_Years > 0).mean())}
    return per, summ


# ════════════════════════════════════════════════════════════ V3
def v3_known_distressed(d):
    deaths = d[d.Death == 1].copy()
    if deaths.empty:
        return pd.DataFrame(), {}
    rows = []
    for root, g in deaths.groupby("ISIN_Root"):
        g = g.sort_values("FY")
        ev = g.Event_Date.dropna().min()
        pre = g[g.Filing_Date <= ev] if pd.notna(ev) else g
        rec = {"ISIN_Root": root, "Company": g.Company.dropna().iat[0] if g.Company.notna().any() else root,
               "Event_Date": ev, "Years_Observed_Before": len(pre)}
        for flag, lab in (("Gate_Zombie", "Zombie"), ("A", "Cond_A"), ("Is_Zombie_State", "ZombieState"),
                          ("AtRisk_or_Worse", "AtRiskPlus")):
            hit = pre[pre[flag].astype(bool)]
            rec[f"Flag_{lab}"] = len(hit) > 0
            rec[f"Lead_Years_{lab}"] = (ev.year + (ev.month > 3) - hit.FY.min()) if len(hit) and pd.notna(ev) else np.nan
        first = {}
        for c in ("G1_ICR_persist", "G2_CashICR_persist", "B", "C", "D"):
            h = pre[pre[c].astype(bool)]
            first[c] = h.FY.min() if len(h) else np.inf
        fmin = min(first.values())
        rec["First_Condition"] = ",".join(k for k, v in first.items() if v == fmin) if np.isfinite(fmin) else "none"
        rec["Max_ZEW_Before"] = pre.ZEW.max()
        rows.append(rec)
    out = pd.DataFrame(rows)
    usable = out[out.Years_Observed_Before >= 3]
    summ = {"deaths": len(out), "deaths_with_3y_history": len(usable),
            "recall_zombie": float(usable.Flag_Zombie.mean()) if len(usable) else np.nan,
            "recall_atrisk_plus": float(usable.Flag_AtRiskPlus.mean()) if len(usable) else np.nan,
            "median_lead_years_zombie": float(usable.Lead_Years_Zombie.median()) if len(usable) else np.nan}
    return out, summ


# ════════════════════════════════════════════════════════════ V4
def v4_sector_balance(d, flag="Gate_Zombie", base=None):
    x = d[d.Gate_Computable] if base is None else base
    tab = pd.crosstab(x.Sector, x[flag].astype(bool))
    if tab.shape[1] < 2 or tab.shape[0] < 2:
        return tab, {}
    chi2, p, dof, _ = chi2_contingency(tab)
    prev = (tab[True] / tab.sum(axis=1)).rename("Prevalence")
    n = tab.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))
    return pd.concat([tab, prev], axis=1).sort_values("Prevalence", ascending=False), \
        {"chi2": chi2, "p_value": p, "dof": dof, "cramers_v": cramers_v}


# ════════════════════════════════════════════════════════════ V5
def v5_jaccard(d, defs=C.DEFINITIONS):
    x = d[d.Gate_Computable]
    M = pd.DataFrame(index=defs, columns=defs, dtype=float)
    for a, b in itertools.product(defs, defs):
        A, B = x[a].astype(bool), x[b].astype(bool)
        u = (A | B).sum()
        M.loc[a, b] = (A & B).sum() / u if u else np.nan
    prev = pd.Series({k: x[k].astype(bool).mean() for k in defs}, name="Prevalence")
    any_ = x[defs].astype(bool).any(axis=1).sum()
    all_ = x[defs].astype(bool).all(axis=1).sum()
    return M, prev, {"any_definition": int(any_), "all_definitions": int(all_),
                     "all_over_any": all_ / any_ if any_ else np.nan}


# ════════════════════════════════════════════════════════════ V6 grids
GRIDS = {
    "icr_cut": [0.8, 0.9, 1.0, 1.1, 1.2],
    "persist": [2, 3, 4],
    "age_min": [0, 5, 10, 15],
    "q_pctl": [0.25, 0.5],
    "eff_pctl": [0.10, 0.25],
    "veto_icr": [2.0, 2.5, 3.0, 3.5, 4.0],
    "veto_nd": [1.5, 2.0, 2.5, 3.0],
}


def v6_grids(d):
    rows = []
    base = d[d.Gate_Computable]
    for p, vals in GRIDS.items():
        for v in vals:
            g = G.run_gate(d, **{p: v})
            z = g.Gate_Zombie[d.Gate_Computable]
            s = g.Gate_Subsidised[d.Gate_Computable]
            rows.append({"Parameter": p, "Value": v, "Zombie_FirmYears": int(z.sum()),
                         "Zombie_Prevalence": float(z.mean()), "Subsidised_FirmYears": int(s.sum()),
                         "Is_Spec_Value": v == getattr(C, {"icr_cut": "ICR_CUT", "persist": "PERSIST_YEARS",
                                                           "age_min": "AGE_MIN", "q_pctl": "Q_PCTL",
                                                           "eff_pctl": "EFF_RATE_PCTL", "veto_icr": "VETO_ICR",
                                                           "veto_nd": "VETO_ND_EBITDA"}[p])})
    return pd.DataFrame(rows)


NOISY = ["EBIT", "CFO", "Interest", "Capex", "Total_Debt", "Revenue", "EBITDA", "MCap_Cr",
         "Net_Debt", "Reserves", "Total_Assets", "Cash_Equivalents", "FCF"]


def v6_bootstrap(raw, d, reps=C.BOOTSTRAP_REPS, sd=C.NOISE_SD, seed=C.SEED):
    """Re-run ratios + gate with ±sd multiplicative noise on accounting inputs; flip rate per firm-year."""
    rng = np.random.default_rng(seed)
    base = d.Gate_Zombie.astype(bool).values
    flips = np.zeros(len(d))
    cols = [c for c in NOISY if c in raw]
    for r in range(reps):
        x = raw.copy()
        for c in cols:
            x[c] = x[c] * rng.normal(1.0, sd, len(x))
        if "FCF" in x and "Capex" in x:
            x["FCF"] = x.CFO - x.Capex
        dd = R.compute(x)
        g = G.run_gate(dd)
        flips += (g.Gate_Zombie.astype(bool).values != base)
    out = d[["ISIN_Root", "FY", "Company", "Gate_Zombie", "ICR", "CashICR"]].copy()
    out["Flip_Rate"] = flips / reps
    return out


# ════════════════════════════════════════════════════════════ with labels
def label_metrics(ml_panel, scores):
    """AUC / PR-AUC / Brier / Youden for each flag or score against the 24-month IBC label."""
    from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score, roc_curve
    y = ml_panel.Label
    rows = []
    if y.nunique() < 2:
        return pd.DataFrame()
    for name, s in scores.items():
        s = pd.to_numeric(s, errors="coerce")
        ok = s.notna() & y.notna()
        if ok.sum() < 20 or y[ok].nunique() < 2:
            continue
        auc = roc_auc_score(y[ok], s[ok])
        ap = average_precision_score(y[ok], s[ok])
        p = (s[ok] - s[ok].min()) / (s[ok].max() - s[ok].min() + 1e-12)
        fpr, tpr, thr = roc_curve(y[ok], s[ok])
        j = np.argmax(tpr - fpr)
        rows.append({"Measure": name, "N": int(ok.sum()), "Events": int(y[ok].sum()), "ROC_AUC": auc,
                     "PR_AUC": ap, "Brier_rescaled": brier_score_loss(y[ok], p),
                     "Youden_J": tpr[j] - fpr[j], "Youden_Threshold": thr[j]})
    return pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False)


**`zew_analysis/survival.py`**

In [ ]:
%%writefile zew_analysis/survival.py
"""
Survival layer — the "death clock" (proposal Objective 4; spec §05 time-to-event).

Clock: years since the firm enters the risk set (first fiscal year the gate can be
evaluated, dated at its filing date = FY end + 2 months). Event = IBC admission
(config.EVENT_COL). Censoring = delisting / merger exit or the data cut-off.

1. Spells without immortal-time bias: every firm contributes a NON-ZOMBIE spell from
   entry until min(zombie entry, end); a firm that becomes a zombie then starts a ZOMBIE
   spell at its first zombie filing date. → Kaplan–Meier + log-rank, for the ZEW state
   and for every one of the 7 definitions.
2. Counting-process panel (one interval per firm-year) → Cox model with time-varying
   covariates (lifelines CoxTimeVaryingFitter): zombie state, severity, size, leverage.
3. Discrete-time hazard (Shumway 2001): logit of an admission in the next 12 months,
   SE clustered by firm.
4. Death-clock summary: median time-to-event, survival at 1/2/3/5 years and restricted
   mean survival time (RMST, 5 y) for zombie vs non-zombie spells.
"""
import numpy as np
import pandas as pd

from . import config as C

YEAR = 365.25


def _end_dates(d):
    firm = d.groupby("ISIN_Root").agg(Event_Date=("Event_Date", "min"), Censor_Date=("Censor_Date", "min"),
                                      Death=("Death", "max"), Last_Filing=("Filing_Date", "max"))
    cutoff = pd.Timestamp(C.DATA_CUTOFF)
    cens = firm.Censor_Date.fillna(cutoff).clip(upper=cutoff)
    # never censor before we stop observing filings + 1 year (keeps late filers in the risk set)
    cens = np.minimum(cens, firm.Last_Filing + pd.Timedelta(days=int(1.5 * YEAR)))
    cens = pd.Series(cens, index=firm.index).clip(upper=cutoff)
    firm["End"] = np.where(firm.Death == 1, firm.Event_Date, cens)
    firm["End"] = pd.to_datetime(firm.End)
    return firm


def spells(d, flag="Is_Zombie_State"):
    """Non-zombie / zombie spells per firm for a given zombie flag column."""
    x = d[d.Gate_Computable & ~d.Data_Insufficient_Firm].sort_values(["ISIN_Root", "FY"])
    firm = _end_dates(d)
    rows = []
    for root, g in x.groupby("ISIN_Root"):
        f = firm.loc[root]
        start = g.Filing_Date.iloc[0]
        end, death = f.End, int(f.Death)
        if pd.isna(end) or end <= start:
            continue
        z = g[g[flag].astype(bool)]
        zstart = z.Filing_Date.iloc[0] if len(z) else None
        if zstart is not None and zstart >= end:
            zstart = None
        stop1 = zstart if zstart is not None else end
        rows.append({"ISIN_Root": root, "Group": "Non-zombie", "Start": start, "Stop": stop1,
                     "Event": death if zstart is None else 0,
                     "T": (stop1 - start).days / YEAR, "Sector": g.Sector.iloc[0],
                     "ZEW_at_start": g.ZEW.iloc[0], "log_TA": g.log_TA.iloc[0]})
        if zstart is not None:
            zr = g[g.Filing_Date == zstart].iloc[0]
            rows.append({"ISIN_Root": root, "Group": "Zombie", "Start": zstart, "Stop": end, "Event": death,
                         "T": (end - zstart).days / YEAR, "Sector": g.Sector.iloc[0],
                         "ZEW_at_start": zr.ZEW, "log_TA": zr.log_TA,
                         "Subsidised": zr.State == "Subsidised Zombie" if "State" in zr else False})
    s = pd.DataFrame(rows)
    if len(s):
        s = s[s["T"] > 0]
    return s


def km_logrank(sp):
    from lifelines import KaplanMeierFitter
    from lifelines.statistics import logrank_test
    out, fits = {}, {}
    for grp, g in sp.groupby("Group"):
        k = KaplanMeierFitter(label=grp).fit(g["T"], g.Event)
        fits[grp] = k
    if {"Zombie", "Non-zombie"} <= set(fits):
        a, b = sp[sp.Group == "Zombie"], sp[sp.Group == "Non-zombie"]
        lr = logrank_test(a["T"], b["T"], a.Event, b.Event)
        out = {"logrank_chi2": lr.test_statistic, "logrank_p": lr.p_value}
    return fits, out


def death_clock(fits, horizon=5.0):
    from lifelines.utils import restricted_mean_survival_time
    rows = []
    for grp, k in fits.items():
        sf = k.survival_function_.iloc[:, 0]
        rec = {"Group": grp, "Spells": int(k.event_observed.size), "Events": int(k.event_observed.sum()),
               "Median_Years": k.median_survival_time_}
        for t in (1, 2, 3, 5):
            rec[f"S({t}y)"] = float(k.survival_function_at_times(t).iloc[0])
        try:
            rec[f"RMST_{int(horizon)}y"] = float(restricted_mean_survival_time(k, t=horizon))
        except Exception:                                           # noqa: BLE001
            rec[f"RMST_{int(horizon)}y"] = np.nan
        rows.append(rec)
    return pd.DataFrame(rows)


# ════════════════════════════════════════════════════════════ counting process
def counting_process(d, covars=("Is_Zombie_State", "Subsidised_State", "ZEW10", "log_TA", "Lev")):
    x = d[d.Gate_Computable & ~d.Data_Insufficient_Firm].sort_values(["ISIN_Root", "FY"]).copy()
    x["Subsidised_State"] = (x.State == "Subsidised Zombie").astype(int)
    x["ZEW10"] = x.ZEW / 10.0
    firm = _end_dates(d)
    x = x.join(firm[["End", "Death"]].rename(columns={"Death": "_death"}), on="ISIN_Root")
    x["Entry"] = x.groupby("ISIN_Root").Filing_Date.transform("min")
    x["Next"] = x.groupby("ISIN_Root").Filing_Date.shift(-1)
    x["stop_date"] = x[["Next", "End"]].min(axis=1)
    x.loc[x.Next.isna(), "stop_date"] = x.End
    x = x[x.stop_date > x.Filing_Date]
    x["start"] = (x.Filing_Date - x.Entry).dt.days / YEAR
    x["stop"] = (x.stop_date - x.Entry).dt.days / YEAR
    x["event"] = ((x._death == 1) & (x.stop_date == x.End)).astype(int)
    cols = ["ISIN_Root", "start", "stop", "event"] + [c for c in covars if c in x]
    cp = x[cols].copy()
    for c in covars:
        if c in cp:
            cp[c] = pd.to_numeric(cp[c], errors="coerce").astype(float)
    return cp.dropna()


def cox_tv(cp, covars):
    from lifelines import CoxTimeVaryingFitter
    use = [c for c in covars if c in cp and cp[c].std() > 0]
    if cp.event.sum() < 3 or not use:
        return None, pd.DataFrame()
    m = CoxTimeVaryingFitter(penalizer=0.01)
    m.fit(cp[["ISIN_Root", "start", "stop", "event"] + use], id_col="ISIN_Root",
          event_col="event", start_col="start", stop_col="stop", show_progress=False)
    s = m.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].copy()
    s.columns = ["HR", "HR_low", "HR_high", "p"]
    return m, s


def cox_by_definition(d, defs=C.DEFINITIONS):
    rows = []
    for dfn in defs:
        if dfn not in d:
            continue
        dd = d.copy()
        dd["Is_Zombie_State"] = dd[dfn].astype(int)
        cp = counting_process(dd, covars=("Is_Zombie_State", "log_TA"))
        _, s = cox_tv(cp, ["Is_Zombie_State", "log_TA"])
        if len(s) and "Is_Zombie_State" in s.index:
            r = s.loc["Is_Zombie_State"]
            rows.append({"Definition": dfn, "HR": r.HR, "HR_low": r.HR_low, "HR_high": r.HR_high, "p": r.p,
                         "Zombie_Intervals": int(cp.Is_Zombie_State.sum()), "Events": int(cp.event.sum())})
    return pd.DataFrame(rows)


# ════════════════════════════════════════════════════════════ discrete-time hazard
def shumway_logit(d, covars=("Is_Zombie_State", "ZEW10", "log_TA", "Lev", "CashICR_c", "Excess_Ret_3y")):
    import statsmodels.api as sm
    x = d[d.Gate_Computable & ~d.Data_Insufficient_Firm].copy()
    x["ZEW10"] = x.ZEW / 10.0
    x["CashICR_c"] = x.CashICR.clip(-10, 10)
    firm = _end_dates(d)
    x = x.join(firm[["End", "Death"]].rename(columns={"Death": "_d"}), on="ISIN_Root")
    x["y"] = ((x._d == 1) & (x.End > x.Filing_Date) &
              (x.End <= x.Filing_Date + pd.Timedelta(days=int(YEAR)))).astype(int)
    x = x[x.End > x.Filing_Date]
    use = [c for c in covars if c in x and pd.to_numeric(x[c], errors="coerce").notna().mean() > 0.5]
    X = x[use].apply(pd.to_numeric, errors="coerce").astype(float)
    ok = X.notna().all(axis=1)
    X, yv, grp = X[ok], x.y[ok], x.ISIN_Root[ok]
    if yv.sum() < 5:
        return None, pd.DataFrame()
    X = sm.add_constant(X, has_constant="add")
    try:
        res = sm.Logit(yv, X).fit(disp=0, cov_type="cluster", cov_kwds={"groups": pd.factorize(grp)[0]})
    except Exception:                                               # noqa: BLE001
        res = sm.Logit(yv, X).fit_regularized(disp=0, alpha=0.01)
    s = pd.DataFrame({"coef": res.params, "OR": np.exp(res.params), "p": getattr(res, "pvalues", np.nan)})
    return res, s


def cox_profiles(sp):
    """Spell-level Cox (group + covariates at spell start) → survival curves for profiles."""
    from lifelines import CoxPHFitter
    x = sp.assign(Zombie=(sp.Group == "Zombie").astype(int)).dropna(subset=["ZEW_at_start", "log_TA"])
    if x.Event.sum() < 3:
        return None, None
    m = CoxPHFitter(penalizer=0.01).fit(x[["T", "Event", "Zombie", "ZEW_at_start", "log_TA"]],
                                        duration_col="T", event_col="Event")
    prof = pd.DataFrame({"Zombie": [0, 0, 1, 1],
                         "ZEW_at_start": [30, 55, 70, 85],
                         "log_TA": [x.log_TA.median()] * 4},
                        index=["Healthy (ZEW 30)", "Watch (ZEW 55)", "Zombie (ZEW 70)", "Severe zombie (ZEW 85)"])
    sf = m.predict_survival_function(prof, times=np.linspace(0, 8, 81))
    sf.columns = prof.index
    return m, sf


**`zew_analysis/ml.py`**

In [ ]:
%%writefile zew_analysis/ml.py
"""
Predictive layer — probability of default (proposal Objective 3).

Unit: firm-year, observed at the filing date (FY end + 2 months; no look-ahead).
Label: IBC admission within PD_HORIZON_MONTHS (24) after the filing date.
       Firm-years whose 24-month window runs past the data cut-off without an event
       are dropped (label incomplete). Firm-years after the event are already removed.
Split: train FY ≤ TRAIN_LAST_FY, test FY ≥ TEST_FIRST_FY (out-of-time, no leakage).
Models: logistic regression (benchmark), XGBoost with scale_pos_weight, and XGBoost
        trained on SMOTE-resampled data (robustness; resampling on the training set only).
Explanation: XGBoost exact TreeSHAP contributions (pred_contribs) — no extra package.
"""
import numpy as np
import pandas as pd

from . import config as C

FEATURES = [
    # levels
    "ICR", "CashICR", "DSCR", "ND_EBITDA", "FCF_Margin_3y", "CFO_to_EBITDA", "Sloan_Accruals",
    "DSO_Index", "ND_Assets", "ST_Debt_Share", "Debt_EBITDA_Div", "EffRate_Gap", "ROCE_minus_WACC",
    "EBITDA_Margin", "RE_TA", "Merton_DD", "Q_Demeaned", "Excess_Ret_3y", "log_TA", "Lev", "Pledge_Pct",
    # velocity (Objective 2)
    "d_ICR", "dd_ICR", "d_CashICR", "d_DE", "dd_DE", "d_ND_Assets", "d_EBITDA_Margin", "WC_Burn",
    "d_CFO_TA", "dlog_Debt", "EBITDA_Margin_Slope",
    # framework outputs
    "ZEW", "P1", "P2", "P3", "P4", "P5", "mu_Z", "Altman_Z", "Beneish_M", "Piotroski_F",
    "A", "B", "C", "D", "Is_Zombie_State",
]


def build_panel(d):
    x = d[~d.Data_Insufficient_Firm].copy()
    horizon = pd.DateOffset(months=C.PD_HORIZON_MONTHS)
    ev = x.Event_Date.where(x.Death == 1)
    end_win = x.Filing_Date + horizon
    x["Label"] = ((ev > x.Filing_Date) & (ev <= end_win)).astype(int)
    complete = (end_win <= pd.Timestamp(C.DATA_CUTOFF)) | (x.Label == 1)
    alive = ev.isna() | (ev > x.Filing_Date)
    cens = x.Censor_Date
    before_exit = cens.isna() | (cens > x.Filing_Date) | (x.Death == 1)
    x = x[complete & alive & before_exit].copy()
    feats = [f for f in FEATURES if f in x]
    for f in feats:
        x[f] = pd.to_numeric(x[f], errors="coerce").astype(float)
        if f in ("ICR", "CashICR", "DSCR", "ND_EBITDA"):
            x[f] = x[f].clip(-50, 99)
    x[feats] = x[feats].replace([np.inf, -np.inf], np.nan)
    return x, feats


def split(x):
    tr = x[x.FY <= C.TRAIN_LAST_FY]
    te = x[x.FY >= C.TEST_FIRST_FY]
    return tr, te


def _metrics(y, p, name):
    from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
    if y.nunique() < 2:
        return {"Model": name, "N": len(y), "Events": int(y.sum())}
    order = np.argsort(-p)
    top = int(max(1, round(0.1 * len(p))))
    cap = y.values[order][:top].sum() / max(1, y.sum())
    from scipy.stats import ks_2samp
    ks = ks_2samp(p[y.values == 1], p[y.values == 0]).statistic
    return {"Model": name, "N": len(y), "Events": int(y.sum()), "ROC_AUC": roc_auc_score(y, p),
            "PR_AUC": average_precision_score(y, p),
            "Brier": brier_score_loss(y, p) if name.startswith("PD_") else np.nan, "KS": ks,
            "Top10pct_Capture": cap}


def fit_all(x, feats, seed=C.SEED):
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    import xgboost as xgb

    tr, te = split(x)
    res = {"train_n": len(tr), "test_n": len(te), "train_events": int(tr.Label.sum()),
           "test_events": int(te.Label.sum())}
    if tr.Label.sum() < 5:
        res["error"] = "fewer than 5 events in training window — model not fitted"
        return res
    Xtr, ytr, Xte, yte = tr[feats], tr.Label, te[feats], te.Label
    pos = max(1, ytr.sum())
    spw = (len(ytr) - pos) / pos

    logit = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                          LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000))
    logit.fit(Xtr, ytr)

    params = dict(n_estimators=400, max_depth=3, learning_rate=0.03, subsample=0.8,
                  colsample_bytree=0.7, min_child_weight=5, reg_lambda=2.0,
                  eval_metric="aucpr", random_state=seed, n_jobs=4)
    xgbm = xgb.XGBClassifier(scale_pos_weight=spw, **params)
    xgbm.fit(Xtr, ytr)

    smote_model = None
    try:
        from imblearn.over_sampling import SMOTE
        imp = SimpleImputer(strategy="median").fit(Xtr)
        k = int(min(5, pos - 1))
        if k >= 1:
            Xs, ys = SMOTE(random_state=seed, k_neighbors=k).fit_resample(imp.transform(Xtr), ytr)
            smote_model = (imp, xgb.XGBClassifier(**params).fit(Xs, ys))
    except Exception as e:                                          # noqa: BLE001
        res["smote_note"] = str(e)

    preds = pd.DataFrame(index=te.index)
    preds["Label"] = yte
    preds["PD_Logit"] = logit.predict_proba(Xte)[:, 1]
    preds["PD_XGB"] = xgbm.predict_proba(Xte)[:, 1]
    if smote_model:
        preds["PD_XGB_SMOTE"] = smote_model[1].predict_proba(smote_model[0].transform(Xte))[:, 1]
    # benchmarks as scores (higher = riskier)
    for name, col, sign in (("ZEW severity", "ZEW", 1), ("Fuzzy μZ", "mu_Z", 1),
                            ("Altman Z″ (−)", "Altman_Z", -1), ("Merton PD", "Merton_DD", -1)):
        if col in te:
            preds[name] = sign * te[col]
    rows = []
    for c in preds.columns.drop("Label"):
        ok = preds[c].notna()
        rows.append(_metrics(preds.Label[ok], preds[c][ok].values, c))
    res["metrics"] = pd.DataFrame(rows)
    res["preds"] = preds
    res["models"] = {"logit": logit, "xgb": xgbm}

    # TreeSHAP
    dm = xgb.DMatrix(Xte, feature_names=feats)
    contrib = xgbm.get_booster().predict(dm, pred_contribs=True)[:, :-1]
    shap = pd.DataFrame(contrib, columns=feats, index=te.index)
    res["shap"] = shap
    res["shap_importance"] = shap.abs().mean().sort_values(ascending=False).rename("Mean_abs_SHAP")
    res["gain_importance"] = pd.Series(xgbm.get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
    res["X_test"] = Xte
    return res


def score_latest(d, res, feats):
    """PD for every firm's latest fiscal year (the dashboard table)."""
    if "models" not in res:
        return pd.DataFrame()
    last = d[~d.Data_Insufficient_Firm].sort_values("FY").groupby("ISIN_Root").tail(1).copy()
    last = last[(last.FY >= d.FY.max() - 1) & (last.Death == 0)]      # live firms, latest filing
    X = last[feats].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    for f in ("ICR", "CashICR", "DSCR", "ND_EBITDA"):
        if f in X:
            X[f] = X[f].clip(-50, 99)
    last["PD_24m_XGB"] = res["models"]["xgb"].predict_proba(X)[:, 1]
    last["PD_24m_Logit"] = res["models"]["logit"].predict_proba(X)[:, 1]
    keep = ["Company", "NSE_Symbol", "Sector", "FY", "State", "Gate_Label", "ZEW", "Top_Pillar",
            "Trajectory", "ICR", "CashICR", "Altman_Z", "Beneish_M", "Piotroski_F", "mu_Z",
            "PD_24m_XGB", "PD_24m_Logit", "Death"]
    return last[[k for k in keep if k in last]].sort_values("PD_24m_XGB", ascending=False)


**`zew_analysis/forensic.py`**

In [ ]:
%%writefile zew_analysis/forensic.py
"""
Forensic layer — the paper's research question (spec §01, reason 3):
"Is zombie status — under any definition — associated with earnings manipulation?"

  1. Earnings-quality profile by state: Beneish M, Sloan accruals, DSRI, TATA,
     share above the Beneish cut-off; Mann–Whitney zombie vs. healthy.
  2. Panel regressions:  M_it  (and Accruals_it) = β·Zombie_it + controls + sector FE + year FE,
     SE clustered by firm — run once per zombie definition (robust if β survives across
     definitions). Dose–response: same model with the fuzzy μ_Z instead of the dummy.
  3. Event time: mean accruals / M-score in years −3…+3 around zombie entry, 95 % CI.
  4. The G2 incremental set: firms flagged ONLY via cash coverage (EBIT says fine, cash says
     not) vs. AAM18 zombies vs. healthy — the spec's predicted earnings-quality gap.
Associational, not causal (spec §11).
"""
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

from . import config as CFG

EQ_VARS = ["Beneish_M", "Sloan_Accruals", "Ben_DSRI", "Ben_TATA", "Ben_AQI", "CFO_to_EBITDA"]


def profile_by_state(d):
    x = d[d.State.isin(CFG.STATE_ORDER[:6])]
    g = x.groupby("State")
    tab = g[[v for v in EQ_VARS if v in x]].median()
    tab["Beneish_Flag_Share"] = g.Beneish_Flag.mean()
    tab["N"] = g.size()
    tab = tab.reindex([s for s in CFG.STATE_ORDER if s in tab.index])
    tests = []
    z = x[x.Is_Zombie_State]
    h = x[x.State == "Healthy"]
    for v in [v for v in EQ_VARS if v in x]:
        a, b = z[v].dropna(), h[v].dropna()
        if len(a) >= 5 and len(b) >= 5:
            st, p = mannwhitneyu(a, b, alternative="two-sided")
            tests.append({"Variable": v, "Zombie_median": a.median(), "Healthy_median": b.median(),
                          "MannWhitney_U": st, "p": p, "n_zombie": len(a), "n_healthy": len(b)})
    return tab, pd.DataFrame(tests)


def _ols_fe(x, y, treat, controls=("log_TA", "Lev")):
    import statsmodels.formula.api as smf
    cols = [y, treat, *controls, "Sector", "FY", "ISIN_Root"]
    z = x[cols].dropna()
    if len(z) < 50 or z[treat].nunique() < 2:
        return None
    z = z.copy()
    lo, hi = z[y].quantile([0.01, 0.99])
    z[y] = z[y].clip(lo, hi)
    f = f"{y} ~ {treat} + " + " + ".join(controls) + " + C(Sector) + C(FY)"
    try:
        m = smf.ols(f, data=z).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(z.ISIN_Root)[0]})
    except Exception:                                               # noqa: BLE001
        return None
    return {"coef": m.params[treat], "se": m.bse[treat], "p": m.pvalues[treat], "n": int(m.nobs),
            "r2": m.rsquared}


def regressions(d):
    x = d[d.Gate_Computable].copy()
    rows = []
    for y in ("Beneish_M", "Sloan_Accruals"):
        if y not in x:
            continue
        for dfn in CFG.DEFINITIONS + ["mu_Z"]:
            if dfn not in x:
                continue
            x["_T"] = pd.to_numeric(x[dfn], errors="coerce").astype(float)
            r = _ols_fe(x, y, "_T")
            if r:
                rows.append({"Outcome": y, "Treatment": dfn, **r})
    return pd.DataFrame(rows)


def event_time(d, var="Sloan_Accruals", window=3):
    x = d.sort_values(["ISIN_Root", "FY"])
    entry = x[x.Zombie_Entry].groupby("ISIN_Root").FY.min()
    x = x[x.ISIN_Root.isin(entry.index)].copy()
    x["k"] = x.FY - x.ISIN_Root.map(entry)
    x = x[x.k.between(-window, window)]
    g = x.groupby("k")[var]
    out = pd.DataFrame({"mean": g.mean(), "sd": g.std(), "n": g.count()})
    out["ci"] = 1.96 * out.sd / np.sqrt(out.n.clip(lower=1))
    return out.reset_index()


def incremental_set(d):
    x = d[d.Gate_Computable].copy()
    x["Group"] = np.select(
        [x.Gate_Zombie & x.Incremental_CashICR, x.AAM18, x.State == "Healthy"],
        ["Cash-coverage-only zombie (G2)", "AAM18 zombie (EBIT-based)", "Healthy"], None)
    x = x[x.Group.notna()]
    tab = x.groupby("Group")[[v for v in ("Sloan_Accruals", "Beneish_M", "Ben_DSRI", "CFO_to_EBITDA",
                                          "ICR", "CashICR") if v in x]].median()
    tab["N_firm_years"] = x.groupby("Group").size()
    return tab, x


**`zew_analysis/plots.py`**

In [ ]:
%%writefile zew_analysis/plots.py
"""
Every chart for the thesis / paper. Each function saves a PNG (200 dpi) to FIG_DIR and
returns its path (or None when the data needed is absent).

Design rules followed throughout (validated reference palette, config.SERIES):
  · categorical hues in fixed order, never cycled; states use the reserved status
    colours and always carry a text label (legend / axis), never colour alone
  · one y-axis per chart (no dual axes); thin 2 px lines; recessive grid; no chart junk
  · sequential = one blue ramp light→dark; diverging = blue ↔ grey ↔ red
"""
import os

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt                                     # noqa: E402
from matplotlib.colors import LinearSegmentedColormap               # noqa: E402

from . import config as C                                           # noqa: E402

SEQ_CMAP = LinearSegmentedColormap.from_list("zew_seq", C.SEQ)
DIV_CMAP = LinearSegmentedColormap.from_list("zew_div", list(C.DIVERGING))


def style():
    plt.rcParams.update({
        "figure.facecolor": C.SURFACE, "axes.facecolor": C.SURFACE, "savefig.facecolor": C.SURFACE,
        "font.family": "DejaVu Sans", "font.size": 9.5, "axes.titlesize": 11, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.labelcolor": C.INK2, "axes.edgecolor": C.GRID,
        "xtick.color": C.INK2, "ytick.color": C.INK2, "text.color": C.INK,
        "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
        "grid.color": C.GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
        "lines.linewidth": 2.0, "legend.frameon": False, "legend.fontsize": 8.5,
        "axes.prop_cycle": matplotlib.cycler(color=C.SERIES),
    })


def _save(fig, name, note=None):
    os.makedirs(C.FIG_DIR, exist_ok=True)
    if note:
        fig.text(0.01, -0.01, note, fontsize=7.5, color=C.MUTED, ha="left", va="top",
                 transform=fig.transFigure)
    p = os.path.join(C.FIG_DIR, name)
    fig.savefig(p, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return p


def _states(present):
    return [s for s in C.STATE_ORDER if s in present]


# ════════════════════════════════════════════════════════════ data
def funnel(funnel_rows, name="F01_attrition_funnel.png"):
    rows = [(s, n) for s, n, _ in funnel_rows] if funnel_rows and len(funnel_rows[0]) == 3 else funnel_rows
    if not rows:
        return None
    lab, val = zip(*rows)
    fig, ax = plt.subplots(figsize=(8, 0.5 * len(rows) + 1))
    y = np.arange(len(rows))[::-1]
    ax.barh(y, val, color=C.SERIES[0], height=0.6)
    for yi, v in zip(y, val):
        ax.text(v, yi, f"  {v:,}", va="center", fontsize=8.5, color=C.INK)
    ax.set_yticks(y, lab)
    ax.set_xlabel("count")
    ax.grid(axis="y", visible=False)
    ax.set_title("Sample construction (Layer 0 attrition funnel)")
    return _save(fig, name)


def events_by_year(d, name="F02_ibc_events_by_year.png"):
    e = d[d.Death == 1].drop_duplicates("ISIN_Root")
    e2 = d[d.Death_Any == 1].drop_duplicates("ISIN_Root")
    if e2.empty:
        return None
    a = e.Event_Date.dt.year.value_counts().sort_index()
    b = e2.Event_Date.dt.year.value_counts().sort_index()
    yrs = sorted(set(a.index) | set(b.index))
    fig, ax = plt.subplots(figsize=(8, 3.6))
    x = np.arange(len(yrs))
    ax.bar(x - 0.2, [b.get(y, 0) for y in yrs], 0.38, color=C.SERIES[1], label="All IBC admissions")
    ax.bar(x + 0.2, [a.get(y, 0) for y in yrs], 0.38, color=C.SERIES[0], label="Substantive (headline)")
    ax.set_xticks(x, [str(int(y)) for y in yrs])
    ax.set_ylabel("firms admitted")
    ax.legend(loc="upper right")
    ax.set_title("IBC admissions in the sample, by calendar year")
    return _save(fig, name, "Substantive = IRP published the CIRP announcement and the case was not closed within a year.")


def coverage_heatmap(d, name="F03_data_coverage.png"):
    fields = ["Revenue", "EBIT", "Interest", "CFO", "Capex", "Total_Debt", "ST_Borrowings",
              "Trade_Receivables", "Cash_Equivalents", "MCap_Cr", "Q", "Merton_DD", "Firm_Age",
              "Pledge_Pct", "ZEW", "Beneish_M", "Altman_Z"]
    fields = [f for f in fields if f in d]
    cov = d.groupby("FY")[fields].apply(lambda g: g.notna().mean()).T
    fig, ax = plt.subplots(figsize=(9, 0.32 * len(fields) + 1.2))
    im = ax.imshow(cov.values, aspect="auto", cmap=SEQ_CMAP, vmin=0, vmax=1)
    ax.set_yticks(range(len(fields)), fields)
    ax.set_xticks(range(cov.shape[1]), cov.columns)
    ax.grid(False)
    for i in range(cov.shape[0]):
        for j in range(cov.shape[1]):
            v = cov.values[i, j]
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=6.5,
                    color="white" if v > 0.6 else C.INK)
    fig.colorbar(im, ax=ax, fraction=0.025, label="share of firm-years non-missing")
    ax.set_title("Data coverage by field and fiscal year")
    return _save(fig, name)


def coverage_distributions(d, name="F04_icr_cashicr_distribution.png"):
    x = d[d.Has_Debt]
    if x.empty:
        return None
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
    bins = np.linspace(-5, 15, 61)
    for ax, col, colr in zip(axes, ("ICR", "CashICR"), (C.SERIES[0], C.SERIES[2])):
        v = x[col].clip(-5, 15).dropna()
        ax.hist(v, bins=bins, color=colr, edgecolor=C.SURFACE, linewidth=0.6)
        ax.axvline(1.0, color=C.STATUS["critical"], lw=1.5, ls="--")
        ax.text(1.1, ax.get_ylim()[1] * 0.92, "break-even = 1", color=C.INK2, fontsize=8)
        share = (x[col] < 1).mean()
        ax.set_title(f"{col}: {share:.1%} of debt-carrying firm-years < 1")
        ax.set_xlabel(f"{col} (clipped to [−5, 15])")
    axes[0].set_ylabel("firm-years")
    return _save(fig, name, "Right-most bar collects every firm-year with coverage ≥ 15×.")


# ════════════════════════════════════════════════════════════ gate
def prevalence_by_year(d, name="F05_gate_labels_by_year.png"):
    x = d[d.Gate_Computable]
    if x.empty:
        return None
    labs = ["Subsidised Zombie", "Zombie", "At Risk"]
    tab = pd.crosstab(x.FY, x.Gate_Label, normalize="index").reindex(columns=labs, fill_value=0) * 100
    fig, ax = plt.subplots(figsize=(8.5, 3.8))
    bottom = np.zeros(len(tab))
    for lab in labs:
        ax.bar(tab.index, tab[lab], bottom=bottom, color=C.STATE_COLORS[lab], label=lab,
               edgecolor=C.SURFACE, linewidth=2, width=0.7,
               hatch="//" if lab == "Subsidised Zombie" else None)
        bottom += tab[lab].values
    ax.set_ylabel("% of classifiable firm-years")
    ax.set_xticks(tab.index)
    ax.legend(loc="upper left", ncol=3)
    ax.set_title("ZEW 4.0 gate outcome by fiscal year")
    return _save(fig, name)


def gate_combinations(d, name="F06_gate_condition_combinations.png"):
    x = d[d.Gate_Computable]
    if x.empty:
        return None
    key = x.A.map({True: "A", False: "·"}) + x.B.map({True: "B", False: "·"}) + \
        x.C.map({True: "C", False: "·"}) + x.D.map({True: "D", False: "·"})
    vc = key[key != "····"].value_counts().head(14)
    fig, ax = plt.subplots(figsize=(8.5, 4))
    colors = [C.STATUS["critical"] if k.startswith("A") and ("B" in k or "C" in k) else C.SERIES[0] for k in vc.index]
    ax.bar(range(len(vc)), vc.values, color=colors, width=0.7)
    ax.set_xticks(range(len(vc)), vc.index, family="monospace")
    for i, v in enumerate(vc.values):
        ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=7.5)
    ax.set_ylabel("firm-years")
    ax.set_title("Which gate conditions fire together (A = debt-service failure, B = low Q, C = cash burn, D = escalator)")
    return _save(fig, name, "Red bars satisfy A ∧ (B ∨ C) = zombie. '·' = condition not met.")


def icr_vs_cashicr(d, name="F07_icr_vs_cashicr.png"):
    x = d[d.Has_Debt & d.ICR.notna() & d.CashICR.notna()]
    if x.empty:
        return None
    xs, ys = x.ICR.clip(-5, 10), x.CashICR.clip(-5, 10)
    inc = x.Incremental_CashICR & x.Gate_Zombie
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    ax.scatter(xs[~inc], ys[~inc], s=8, color=C.SERIES[0], alpha=0.25, linewidths=0, label="other firm-years")
    ax.scatter(xs[inc], ys[inc], s=18, color=C.SERIES[1], edgecolor=C.SURFACE, linewidth=0.8,
               label="zombie only via cash coverage (G2)")
    ax.axvline(1, color=C.MUTED, lw=1, ls="--")
    ax.axhline(1, color=C.MUTED, lw=1, ls="--")
    ax.text(0.98, 0.02, "lower-right quadrant:\nEBIT covers interest, cash does not", color=C.INK2,
            fontsize=8, ha="right", va="bottom", transform=ax.transAxes)
    ax.set_xlabel("ICR = EBIT ÷ interest (clipped)")
    ax.set_ylabel("Cash ICR = CFO ÷ interest (clipped)")
    ax.legend(loc="upper left")
    ax.set_title("Accrual vs cash interest coverage")
    return _save(fig, name)


def sector_prevalence(tab, stats, name="F08_sector_balance.png"):
    if tab is None or len(tab) == 0 or "Prevalence" not in tab:
        return None
    t = tab.sort_values("Prevalence")
    fig, ax = plt.subplots(figsize=(7.5, 0.32 * len(t) + 1.3))
    ax.barh(range(len(t)), t.Prevalence * 100, color=C.SERIES[0], height=0.65)
    ax.set_yticks(range(len(t)), t.index)
    ax.set_xlabel("zombie prevalence, % of classifiable firm-years")
    ax.grid(axis="y", visible=False)
    sub = f"χ² = {stats.get('chi2', np.nan):.1f}, p = {stats.get('p_value', np.nan):.3g}, Cramér's V = {stats.get('cramers_v', np.nan):.2f}" if stats else ""
    ax.set_title("V4 — zombie prevalence by sector")
    return _save(fig, name, sub)


# ════════════════════════════════════════════════════════════ severity & states
def severity_by_state(d, name="F09_severity_by_state.png"):
    x = d[d.ZEW.notna()]
    sts = _states(x.State.unique())
    if not sts:
        return None
    fig, ax = plt.subplots(figsize=(8.5, 4))
    data = [x.loc[x.State == s, "ZEW"].values for s in sts]
    bp = ax.boxplot(data, patch_artist=True, widths=0.55, showfliers=False,
                    medianprops={"color": C.INK, "linewidth": 1.5})
    for patch, s in zip(bp["boxes"], sts):
        patch.set_facecolor(C.STATE_COLORS[s])
        patch.set_edgecolor(C.SURFACE)
    for yv, lab in ((C.BAND_HEALTHY, "40"), (C.BAND_WATCH, "55"), (C.BAND_ATRISK, "70")):
        ax.axhline(yv, color=C.MUTED, lw=0.8, ls=":")
    ax.set_xticks(range(1, len(sts) + 1), sts)
    ax.set_ylabel("ZEW 4.0 severity (0–100)")
    ax.set_ylim(0, 100)
    ax.set_title("Severity score by state (diagnostic layer — does not set the label)")
    return _save(fig, name, "Dotted lines: severity bands 40 / 55 / 70.")


def pillars_by_state(d, name="F10_pillars_by_state.png"):
    ps = [p for p in C.PILLAR_WEIGHTS if p in d]
    x = d[d.ZEW.notna()]
    sts = _states(x.State.unique())
    if not sts or not ps:
        return None
    m = x.groupby("State")[ps].median().reindex(sts)
    fig, ax = plt.subplots(figsize=(6.5, 0.5 * len(sts) + 1.5))
    im = ax.imshow(m.values, cmap=DIV_CMAP, vmin=0, vmax=100, aspect="auto")
    names = {"P1": "P1 Debt service", "P2": "P2 Cash & earnings quality", "P3": "P3 Capital structure",
             "P4": "P4 Profitability", "P5": "P5 Market"}
    ax.set_xticks(range(len(ps)), [names[p] for p in ps], rotation=20, ha="right")
    ax.set_yticks(range(len(sts)), sts)
    ax.grid(False)
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            v = m.values[i, j]
            if v == v:
                ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=8,
                        color="white" if abs(v - 50) > 30 else C.INK)
    fig.colorbar(im, ax=ax, fraction=0.04, label="median pillar score (50 = sector median)")
    ax.set_title("Root cause — median pillar score by state")
    return _save(fig, name)


def trajectories(d, n=6, name="F11_trajectories_ibc_firms.png"):
    deaths = d[d.Death == 1]
    firms = deaths.groupby("ISIN_Root").ZEW.count().sort_values(ascending=False).head(n).index
    if len(firms) == 0:
        firms = d.groupby("ISIN_Root").ZEW.max().sort_values(ascending=False).head(n).index
    if len(firms) == 0:
        return None
    cols = 3
    rows = int(np.ceil(len(firms) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(10, 2.8 * rows), sharey=True, squeeze=False)
    for ax, f in zip(axes.flat, firms):
        g = d[d.ISIN_Root == f].sort_values("FY")
        for lo, hi, s in ((0, 40, "Healthy"), (55, 70, "Watch"), (70, 100, "At Risk")):
            ax.axhspan(lo, hi, color=C.STATE_COLORS[s], alpha=0.08, lw=0)
        ax.plot(g.FY, g.ZEW, color=C.SERIES[0], marker="o", ms=4)
        z = g[g.Is_Zombie_State]
        ax.scatter(z.FY, z.ZEW, s=40, color=C.STATE_COLORS["Zombie"], zorder=3, edgecolor=C.SURFACE,
                   linewidth=1, label="zombie state")
        ev = g.Event_Date.dropna()
        if len(ev):
            fy_ev = ev.iat[0].year + (ev.iat[0].month > 3)
            ax.axvline(fy_ev, color=C.INK, lw=1.2, ls="--")
            ax.text(fy_ev, 96, " IBC", fontsize=7.5, va="top")
        nm = g.Company.dropna().iat[0] if g.Company.notna().any() else f
        ax.set_title(str(nm)[:28], fontsize=9)
        ax.set_ylim(0, 100)
    for ax in list(axes.flat)[len(firms):]:
        ax.set_visible(False)
    axes[0, 0].set_ylabel("ZEW severity")
    fig.suptitle("Severity trajectory before IBC admission (red dots = zombie state)", x=0.01, ha="left",
                 fontweight="bold")
    fig.tight_layout()
    return _save(fig, name)


def transition_heatmap(tm, name="F12_state_transitions.png"):
    if tm is None or tm.empty:
        return None
    fig, ax = plt.subplots(figsize=(7.5, 5.5))
    im = ax.imshow(tm.values, cmap=SEQ_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(tm.shape[1]), tm.columns, rotation=30, ha="right")
    ax.set_yticks(range(tm.shape[0]), tm.index)
    ax.grid(False)
    for i in range(tm.shape[0]):
        for j in range(tm.shape[1]):
            v = tm.values[i, j]
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=7.5,
                    color="white" if v > 0.55 else C.INK)
    ax.set_xlabel("state in year t+1")
    ax.set_ylabel("state in year t")
    fig.colorbar(im, ax=ax, fraction=0.04, label="transition probability")
    ax.set_title("Layer 3 — one-year state transition matrix")
    return _save(fig, name)


def state_share(d, name="F13_state_share_by_year.png"):
    x = d[d.State.notna() & (d.State != "Data Insufficient")]
    if x.empty:
        return None
    sts = _states(x.State.unique())
    tab = pd.crosstab(x.FY, x.State, normalize="index").reindex(columns=sts, fill_value=0) * 100
    fig, ax = plt.subplots(figsize=(8.5, 4))
    ax.stackplot(tab.index, [tab[s] for s in sts], colors=[C.STATE_COLORS[s] for s in sts],
                 labels=sts, edgecolor=C.SURFACE, linewidth=1.5)
    ax.set_ylim(0, 100)
    ax.set_ylabel("% of firms")
    ax.set_xticks(tab.index)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=len(sts))
    ax.set_title("Distribution of firms across states, by fiscal year")
    return _save(fig, name)


# ════════════════════════════════════════════════════════════ benchmarks
def definitions_by_year(d, name="F14_prevalence_by_definition.png"):
    defs = [k for k in C.DEFINITIONS if k in d]
    x = d[d.Gate_Computable]
    if x.empty:
        return None
    tab = x.groupby("FY")[defs].mean() * 100
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ends = []
    for i, k in enumerate(defs):
        ax.plot(tab.index, tab[k], color=C.SERIES[i], lw=2.6 if k == "ZEW4" else 1.8,
                marker="o", ms=3.5, label=k)
        ends.append([tab[k].iloc[-1], k])
    # direct end labels, nudged apart so they never overlap
    ends.sort()
    gap = max(0.6, (tab.max().max() - tab.min().min()) * 0.045)
    for j in range(1, len(ends)):
        ends[j][0] = max(ends[j][0], ends[j - 1][0] + gap)
    for yv, k in ends:
        ax.text(tab.index[-1] + 0.2, yv, k, color=C.INK2, fontsize=7.5, va="center")
    ax.set_ylabel("% of classifiable firm-years flagged")
    ax.set_xticks(tab.index)
    ax.set_xlim(tab.index.min() - 0.3, tab.index.max() + 1.2)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=len(defs))
    ax.set_title("Zombie prevalence under the seven definitions")
    return _save(fig, name)


def jaccard_heatmap(M, extra, name="F15_jaccard_agreement.png"):
    if M is None or M.empty:
        return None
    fig, ax = plt.subplots(figsize=(6.2, 5.2))
    im = ax.imshow(M.values.astype(float), cmap=SEQ_CMAP, vmin=0, vmax=1)
    ax.set_xticks(range(len(M)), M.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(M)), M.index)
    ax.grid(False)
    for i in range(len(M)):
        for j in range(len(M)):
            v = float(M.values[i, j])
            if v == v:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if v > 0.55 else C.INK)
    fig.colorbar(im, ax=ax, fraction=0.04, label="Jaccard index")
    ax.set_title("V5 — agreement between zombie definitions")
    note = f"Flagged by all 7 ÷ flagged by any = {extra.get('all_over_any', np.nan):.1%} (ECB WP 2743 found ≈10 %)." if extra else None
    return _save(fig, name, note)


def altman_by_state(d, name="F16_altman_zone_by_state.png"):
    x = d[d.Altman_Zone.notna()]
    sts = _states(x.State.unique())
    if not sts:
        return None
    tab = pd.crosstab(x.State, x.Altman_Zone, normalize="index").reindex(index=sts).reindex(
        columns=["Safe", "Grey", "Distress"], fill_value=0) * 100
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(sts) + 1.5))
    left = np.zeros(len(tab))
    zc = {"Safe": C.STATUS["good"], "Grey": "#b4b2ab", "Distress": C.STATUS["critical"]}
    for z in tab.columns:
        ax.barh(range(len(tab)), tab[z], left=left, color=zc[z], label=f"Altman {z}",
                edgecolor=C.SURFACE, linewidth=2, height=0.65)
        left += tab[z].values
    ax.set_yticks(range(len(tab)), tab.index)
    ax.set_xlabel("% of firm-years")
    ax.grid(axis="y", visible=False)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3)
    ax.set_title("Altman Z″ (emerging-markets) zone within each ZEW state")
    return _save(fig, name)


# ════════════════════════════════════════════════════════════ validation
def v1_null(v1, name="F17_v1_null_universe.png"):
    if v1 is None or v1.empty:
        return None
    fig, ax = plt.subplots(figsize=(6.5, 3.2))
    cols = [C.SERIES[0], C.SERIES[1]]
    ax.barh(range(len(v1)), v1.Flagged_Pct, color=cols[:len(v1)], height=0.55)
    for i, (p, n) in enumerate(zip(v1.Flagged_Pct, v1.Flagged)):
        ax.text(p, i, f"  {p:.1f}%  ({n} firms)", va="center", fontsize=9)
    ax.set_yticks(range(len(v1)), v1.Method)
    ax.set_xlim(0, max(40, v1.Flagged_Pct.max() * 1.3))
    ax.set_xlabel(f"% of a perfectly healthy universe (n = {int(v1.Firms.iat[0])}) flagged")
    ax.grid(axis="y", visible=False)
    ax.set_title("V1 — null-universe test: absolute gate vs percentile rule")
    return _save(fig, name)


def v3_lead(v3, name="F18_v3_lead_time.png"):
    if v3 is None or v3.empty:
        return None
    u = v3[v3.Years_Observed_Before >= 3]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    rec = [u.Flag_Cond_A.mean(), u.Flag_Zombie.mean(), u.Flag_ZombieState.mean(), u.Flag_AtRiskPlus.mean()]
    labs = ["Condition A", "Gate zombie", "Zombie state", "At Risk or worse"]
    axes[0].bar(range(4), np.array(rec) * 100, color=C.SERIES[0], width=0.6)
    for i, r in enumerate(rec):
        axes[0].text(i, r * 100, f"{r:.0%}", ha="center", va="bottom", fontsize=8.5)
    axes[0].set_xticks(range(4), labs, rotation=15)
    axes[0].set_ylim(0, 105)
    axes[0].set_ylabel("% of IBC firms flagged before admission")
    axes[0].set_title(f"V3 — recall (n = {len(u)} IBC firms, ≥ 3 y history)")
    lt = u.Lead_Years_Zombie.dropna()
    if len(lt):
        axes[1].hist(lt, bins=np.arange(-0.5, lt.max() + 1.5, 1), color=C.SERIES[1], edgecolor=C.SURFACE)
        axes[1].axvline(lt.median(), color=C.INK, ls="--", lw=1.2)
        axes[1].text(lt.median(), axes[1].get_ylim()[1] * 0.9, f"  median {lt.median():.0f} y", fontsize=8.5)
    axes[1].set_xlabel("years between first zombie flag and IBC admission")
    axes[1].set_ylabel("firms")
    axes[1].set_title("Lead time")
    fig.tight_layout()
    return _save(fig, name)


def v6_grid(grid, name="F19_v6_threshold_stability.png"):
    if grid is None or grid.empty:
        return None
    ps = grid.Parameter.unique()
    cols = 4
    rows = int(np.ceil(len(ps) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(11, 2.7 * rows), squeeze=False)
    nice = {"icr_cut": "ICR cut", "persist": "persistence (years)", "age_min": "minimum age",
            "q_pctl": "Q percentile", "eff_pctl": "eff.-rate percentile", "veto_icr": "veto ICR",
            "veto_nd": "veto ND/EBITDA"}
    for ax, p in zip(axes.flat, ps):
        g = grid[grid.Parameter == p]
        ax.plot(g.Value, g.Zombie_Prevalence * 100, color=C.SERIES[0], marker="o", ms=5)
        s = g[g.Is_Spec_Value]
        ax.scatter(s.Value, s.Zombie_Prevalence * 100, s=70, color=C.SERIES[1], zorder=3,
                   edgecolor=C.SURFACE, linewidth=1.5, label="spec value")
        ax.set_title(nice.get(p, p), fontsize=9.5)
        ax.set_ylim(bottom=0)
    for ax in list(axes.flat)[len(ps):]:
        ax.set_visible(False)
    axes[0, 0].set_ylabel("zombie prevalence %")
    axes[0, 0].legend(loc="lower right")
    fig.suptitle("V6 — threshold stability: plateaus, not cliffs (orange = specified value)",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    return _save(fig, name)


def flip_rates(boot, name="F20_v6_bootstrap_flip_rates.png"):
    if boot is None or boot.empty:
        return None
    v = boot.Flip_Rate[boot.Flip_Rate > 0]
    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.hist(v, bins=np.linspace(0, 1, 21), color=C.SERIES[0], edgecolor=C.SURFACE)
    ax.set_xlabel("share of noisy replications in which the label flips")
    ax.set_ylabel("firm-years")
    stable = 1 - (boot.Flip_Rate > 0).mean()
    ax.set_title(f"V6 — bootstrap flip rates (±{C.NOISE_SD:.0%} noise): {stable:.1%} of firm-years never flip")
    return _save(fig, name)


def fuzzy_distribution(d, name="F21_fuzzy_membership.png"):
    x = d[d.mu_Z.notna() & d.Gate_Computable]
    if x.empty:
        return None
    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    bins = np.linspace(0, 1, 21)
    ax.hist(x.loc[~x.Gate_Zombie, "mu_Z"], bins=bins, color=C.SERIES[0], label="crisp: not zombie",
            edgecolor=C.SURFACE)
    ax.hist(x.loc[x.Gate_Zombie, "mu_Z"], bins=bins, color=C.SERIES[1], label="crisp: zombie",
            edgecolor=C.SURFACE, alpha=0.9)
    ax.set_yscale("log")
    ax.set_xlabel("fuzzy zombie membership μZ")
    ax.set_ylabel("firm-years (log scale)")
    ax.legend()
    ax.set_title("Fuzzy variant — membership vs crisp classification")
    return _save(fig, name)


# ════════════════════════════════════════════════════════════ survival
def _tmax(fits, min_at_risk=10):
    """Plot only while ≥ min_at_risk spells remain in every group (avoids noisy tails)."""
    ts = []
    for k in fits.values():
        at = k.event_table.at_risk
        ok = at[at >= min_at_risk]
        ts.append(ok.index.max() if len(ok) else at.index.max())
    return max(1.0, min(ts)) if ts else None


def km_plot(fits, lr, name="F22_km_zombie_vs_nonzombie.png", title=None):
    if not fits:
        return None
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    order = [g for g in ("Non-zombie", "Zombie") if g in fits]
    colors = {"Non-zombie": C.SERIES[0], "Zombie": C.STATUS["critical"]}
    for g in order:
        fits[g].plot_survival_function(ax=ax, ci_show=True, color=colors[g], lw=2)
    ax.set_xlim(0, _tmax(fits))
    ax.set_xlabel("years since entering the risk set / zombie entry")
    ax.set_ylabel("probability of no IBC admission")
    ax.set_ylim(0, 1.02)
    try:
        from lifelines.plotting import add_at_risk_counts
        add_at_risk_counts(*[fits[g] for g in order], ax=ax, rows_to_show=["At risk", "Events"])
    except Exception:                                               # noqa: BLE001
        pass
    sub = f"log-rank χ² = {lr.get('logrank_chi2', np.nan):.1f}, p = {lr.get('logrank_p', np.nan):.2g}" if lr else ""
    ax.set_title(title or "Kaplan–Meier survival: zombie vs non-zombie spells")
    ax.legend(loc="lower left")
    return _save(fig, name, sub)


def km_by_definition(km_defs, name="F23_km_by_definition.png"):
    if not km_defs:
        return None
    n = len(km_defs)
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 3 * rows), sharey=True, squeeze=False)
    for ax, (dfn, (fits, lr)) in zip(axes.flat, km_defs.items()):
        for g, colr in (("Non-zombie", C.SERIES[0]), ("Zombie", C.STATUS["critical"])):
            if g in fits:
                sf = fits[g].survival_function_
                ax.step(sf.index, sf.iloc[:, 0], where="post", color=colr, lw=1.8, label=g)
        p = lr.get("logrank_p", np.nan) if lr else np.nan
        ax.set_xlim(0, _tmax(fits))
        ax.set_title(f"{dfn}  (log-rank p = {p:.2g})", fontsize=9)
        ax.set_ylim(0, 1.02)
    for ax in list(axes.flat)[n:]:
        ax.set_visible(False)
    axes[0, 0].legend(loc="lower left")
    axes[0, 0].set_ylabel("survival")
    fig.suptitle("Kaplan–Meier by zombie definition — does the death clock depend on the definition?",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    return _save(fig, name)


def forest(cox_defs, name="F24_cox_hazard_ratios.png", title=None):
    if cox_defs is None or cox_defs.empty:
        return None
    t = cox_defs.iloc[::-1]
    lab = t["Definition"] if "Definition" in t else t.index
    fig, ax = plt.subplots(figsize=(7, 0.45 * len(t) + 1.4))
    y = np.arange(len(t))
    ax.hlines(y, t.HR_low, t.HR_high, color=C.SERIES[0], lw=2)
    ax.scatter(t.HR, y, s=50, color=C.SERIES[0], zorder=3, edgecolor=C.SURFACE, linewidth=1.5)
    ax.axvline(1, color=C.MUTED, ls="--", lw=1)
    ax.set_xscale("log")
    ax.set_yticks(y, lab)
    for yi, (hr, p) in enumerate(zip(t.HR, t.p)):
        ax.text(ax.get_xlim()[1], yi, f" HR {hr:.2f}  p={p:.2g}", va="center", fontsize=7.5, color=C.INK2)
    ax.set_xlabel("hazard ratio of IBC admission (log scale, 95 % CI)")
    ax.grid(axis="y", visible=False)
    ax.set_title(title or "Cox (time-varying) — zombie hazard ratio under each definition")
    return _save(fig, name, "Each row: separate model with the definition's zombie dummy + log total assets.")


def cox_profile_curves(sf, name="F25_cox_survival_profiles.png"):
    if sf is None:
        return None
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    cols = [C.STATE_COLORS["Healthy"], C.STATE_COLORS["Watch"], C.STATE_COLORS["Zombie"],
            C.STATE_COLORS["Subsidised Zombie"]]
    for c, colr in zip(sf.columns, cols):
        ax.plot(sf.index, sf[c], color=colr, label=c)
        ax.text(sf.index[-1], sf[c].iloc[-1], f" {c}", fontsize=7.5, va="center", color=C.INK2)
    ax.set_xlim(0, sf.index.max() * 1.3)
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("years")
    ax.set_ylabel("predicted probability of no IBC admission")
    ax.legend(loc="lower left")
    ax.set_title("The 'death clock' — Cox-predicted survival for four firm profiles")
    return _save(fig, name)


# ════════════════════════════════════════════════════════════ ML
_MODEL_ORDER = ["PD_XGB", "PD_Logit", "ZEW severity", "Altman Z″ (−)", "Fuzzy μZ"]


def roc_pr(preds, name="F26_roc_pr_curves.png"):
    from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve, average_precision_score
    if preds is None or preds.Label.nunique() < 2:
        return None
    ms = [m for m in _MODEL_ORDER if m in preds]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
    for i, m in enumerate(ms):
        ok = preds[m].notna()
        y, p = preds.Label[ok], preds[m][ok]
        fpr, tpr, _ = roc_curve(y, p)
        axes[0].plot(fpr, tpr, color=C.SERIES[i], lw=2.4 if i == 0 else 1.6,
                     label=f"{m} (AUC {roc_auc_score(y, p):.2f})")
        pr, rc, _ = precision_recall_curve(y, p)
        axes[1].plot(rc, pr, color=C.SERIES[i], lw=2.4 if i == 0 else 1.6,
                     label=f"{m} (AP {average_precision_score(y, p):.2f})")
    axes[0].plot([0, 1], [0, 1], color=C.MUTED, ls="--", lw=1)
    axes[0].set_xlabel("false-positive rate")
    axes[0].set_ylabel("true-positive rate")
    axes[0].set_title("ROC — out-of-time test set")
    axes[0].legend(loc="lower right", fontsize=7.5)
    base = preds.Label.mean()
    axes[1].axhline(base, color=C.MUTED, ls="--", lw=1)
    axes[1].set_xlabel("recall")
    axes[1].set_ylabel("precision")
    axes[1].set_title(f"Precision–recall (base rate {base:.1%})")
    axes[1].legend(loc="upper right", fontsize=7.5)
    fig.tight_layout()
    return _save(fig, name)


def calibration(preds, name="F27_calibration.png"):
    if preds is None or preds.Label.nunique() < 2:
        return None
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    for i, m in enumerate([m for m in ("PD_XGB", "PD_Logit", "PD_XGB_SMOTE") if m in preds]):
        q = pd.qcut(preds[m].rank(method="first"), 10, labels=False)
        g = preds.groupby(q).agg(p=(m, "mean"), y=("Label", "mean"))
        ax.plot(g.p, g.y, marker="o", ms=5, color=C.SERIES[i], label=m)
    lim = max(0.05, preds[[c for c in ("PD_XGB", "PD_Logit") if c in preds]].max().max())
    ax.plot([0, lim], [0, lim], color=C.MUTED, ls="--", lw=1)
    ax.set_xlabel("mean predicted PD (decile)")
    ax.set_ylabel("observed IBC rate (decile)")
    ax.legend()
    ax.set_title("Calibration by decile")
    return _save(fig, name, "Balanced class weights inflate raw PDs — rank metrics are the primary yardstick.")


def shap_summary(res, top=20, name="F28_shap_importance.png"):
    if not res or "shap" not in res:
        return None
    imp = res["shap_importance"].head(top)[::-1]
    sh, X = res["shap"], res["X_test"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 0.32 * len(imp) + 1.5), gridspec_kw={"width_ratios": [1, 1.4]})
    axes[0].barh(range(len(imp)), imp.values, color=C.SERIES[0], height=0.65)
    axes[0].set_yticks(range(len(imp)), imp.index)
    axes[0].set_xlabel("mean |SHAP| (log-odds)")
    axes[0].grid(axis="y", visible=False)
    axes[0].set_title("Top drivers of predicted IBC risk")
    for i, f in enumerate(imp.index):
        v = X[f]
        r = (v.rank(pct=True)).fillna(0.5)
        axes[1].scatter(sh[f], np.full(len(sh), i) + np.random.default_rng(0).uniform(-0.25, 0.25, len(sh)),
                        c=r, cmap=DIV_CMAP, s=6, alpha=0.7, linewidths=0)
    axes[1].axvline(0, color=C.MUTED, lw=1)
    axes[1].set_yticks(range(len(imp)), imp.index)
    axes[1].set_xlabel("SHAP value (→ higher risk)")
    axes[1].set_title("Direction (blue = low feature value, red = high)")
    fig.tight_layout()
    return _save(fig, name)


def confusion(preds, model="PD_XGB", name="F29_confusion_matrix.png"):
    from sklearn.metrics import confusion_matrix, roc_curve
    if preds is None or model not in preds or preds.Label.nunique() < 2:
        return None
    fpr, tpr, thr = roc_curve(preds.Label, preds[model])
    t = thr[np.argmax(tpr - fpr)]
    cm = confusion_matrix(preds.Label, preds[model] >= t)
    fig, ax = plt.subplots(figsize=(4.6, 4))
    ax.imshow(cm, cmap=SEQ_CMAP)
    labs = ["no IBC", "IBC ≤ 24 m"]
    ax.set_xticks([0, 1], [f"pred. {l}" for l in labs])
    ax.set_yticks([0, 1], [f"actual {l}" for l in labs])
    ax.grid(False)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=11,
                    color="white" if cm[i, j] > cm.max() / 2 else C.INK)
    ax.set_title(f"{model} at Youden threshold {t:.3f}")
    return _save(fig, name)


def pd_distribution(preds, model="PD_XGB", name="F30_pd_distribution.png"):
    if preds is None or model not in preds or preds.Label.nunique() < 2:
        return None
    fig, ax = plt.subplots(figsize=(7, 3.6))
    bins = np.linspace(0, 1, 26)
    ax.hist(preds.loc[preds.Label == 0, model], bins=bins, color=C.SERIES[0], label="no IBC within 24 m",
            edgecolor=C.SURFACE, density=True)
    ax.hist(preds.loc[preds.Label == 1, model], bins=bins, color=C.SERIES[1], label="IBC within 24 m",
            edgecolor=C.SURFACE, density=True, alpha=0.85)
    ax.set_xlabel("predicted PD")
    ax.set_ylabel("density")
    ax.legend()
    ax.set_title("Separation of predicted PD by outcome (test set)")
    return _save(fig, name)


# ════════════════════════════════════════════════════════════ forensic
def beneish_by_state(d, name="F31_beneish_by_state.png"):
    x = d[d.Beneish_M.notna()]
    sts = _states(x.State.unique())
    sts = [s for s in sts if s != "Data Insufficient"]
    if not sts:
        return None
    fig, ax = plt.subplots(figsize=(8.5, 4))
    data = [x.loc[x.State == s, "Beneish_M"].clip(-6, 3).values for s in sts]
    bp = ax.boxplot(data, patch_artist=True, widths=0.55, showfliers=False,
                    medianprops={"color": C.INK, "linewidth": 1.5})
    for patch, s in zip(bp["boxes"], sts):
        patch.set_facecolor(C.STATE_COLORS[s])
        patch.set_edgecolor(C.SURFACE)
    ax.axhline(C.BENEISH_CUT, color=C.INK, ls="--", lw=1.2)
    ax.text(len(sts) + 0.45, C.BENEISH_CUT, f"cut-off {C.BENEISH_CUT}", va="center", fontsize=8)
    ax.set_xticks(range(1, len(sts) + 1), sts)
    ax.set_ylabel("Beneish M-score (clipped)")
    ax.set_title("Earnings-manipulation likelihood by state")
    share = x.groupby("State").Beneish_Flag.mean().reindex(sts)
    for i, s in enumerate(share, 1):
        ax.text(i, ax.get_ylim()[0] + 0.2, f"{s:.0%} > cut", ha="center", fontsize=7.5, color=C.INK2)
    return _save(fig, name)


def event_time_plot(et_acc, et_m, name="F32_event_time_zombie_entry.png"):
    if (et_acc is None or et_acc.empty) and (et_m is None or et_m.empty):
        return None
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for ax, et, lab, colr in ((axes[0], et_acc, "Sloan accruals ÷ assets", C.SERIES[0]),
                              (axes[1], et_m, "Beneish M-score", C.SERIES[1])):
        if et is None or et.empty:
            ax.set_visible(False)
            continue
        ax.fill_between(et.k, et["mean"] - et.ci, et["mean"] + et.ci, color=colr, alpha=0.15, lw=0)
        ax.plot(et.k, et["mean"], color=colr, marker="o", ms=5)
        ax.axvline(0, color=C.INK, ls="--", lw=1)
        ax.set_xlabel("years relative to zombie entry")
        ax.set_ylabel(lab)
        ax.set_title(lab)
    fig.suptitle("Event time — earnings quality around zombie entry (mean, 95 % CI)", x=0.01, ha="left",
                 fontweight="bold")
    fig.tight_layout()
    return _save(fig, name)


def incremental_plot(x, name="F33_incremental_cash_coverage_set.png"):
    if x is None or x.empty:
        return None
    groups = ["Healthy", "AAM18 zombie (EBIT-based)", "Cash-coverage-only zombie (G2)"]
    groups = [g for g in groups if g in set(x.Group)]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, v, lab in ((axes[0], "Sloan_Accruals", "Sloan accruals ÷ assets"),
                       (axes[1], "Beneish_M", "Beneish M-score")):
        if v not in x:
            ax.set_visible(False)
            continue
        data = [x.loc[x.Group == g, v].dropna().clip(*x[v].quantile([0.02, 0.98])).values for g in groups]
        bp = ax.boxplot(data, patch_artist=True, widths=0.55, showfliers=False,
                        medianprops={"color": C.INK, "linewidth": 1.5})
        for patch, colr in zip(bp["boxes"], [C.STATE_COLORS["Healthy"], C.SERIES[0], C.SERIES[1]]):
            patch.set_facecolor(colr)
            patch.set_edgecolor(C.SURFACE)
        ax.set_xticks(range(1, len(groups) + 1), [g.replace(" (", "\n(") for g in groups], fontsize=8)
        ax.set_title(lab)
    fig.suptitle("G2 incremental set — zombies that EBIT-based coverage misses", x=0.01, ha="left",
                 fontweight="bold")
    fig.tight_layout()
    return _save(fig, name)


def pledge_by_state(d, name="F34_promoter_pledge_by_state.png"):
    x = d[d.Pledge_Pct.notna()]
    sts = [s for s in _states(x.State.unique()) if s != "Data Insufficient"]
    if len(sts) < 2:
        return None
    fig, ax = plt.subplots(figsize=(8, 3.6))
    med = x.groupby("State").Pledge_Pct.agg(["mean", "count"]).reindex(sts)
    ax.bar(range(len(sts)), med["mean"], color=[C.STATE_COLORS[s] for s in sts], width=0.6)
    for i, (m, n) in enumerate(zip(med["mean"], med["count"])):
        ax.text(i, m, f"{m:.1f}%\n(n={n})", ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(range(len(sts)), sts)
    ax.set_ylabel("mean % of promoter shares pledged")
    ax.set_title("India diagnostic — promoter pledge by state")
    return _save(fig, name, "Pledge data cover FY2022+ only (NSE shareholding XBRL); diagnostic layer, never in the label.")


def risk_table_fig(tab, n=20, name="F35_top_risk_firms.png"):
    if tab is None or tab.empty:
        return None
    t = tab.head(n).copy()
    fig, ax = plt.subplots(figsize=(11, 0.32 * len(t) + 1.2))
    ax.axis("off")
    cols = [c for c in ("Company", "Sector", "FY", "State", "ZEW", "Top_Pillar", "ICR", "CashICR",
                        "Altman_Z", "PD_24m_XGB") if c in t]
    cell = t[cols].copy()
    for c in ("ZEW", "ICR", "CashICR", "Altman_Z"):
        if c in cell:
            cell[c] = cell[c].map(lambda v: f"{v:.1f}" if v == v else "–")
    if "PD_24m_XGB" in cell:
        cell["PD_24m_XGB"] = cell.PD_24m_XGB.map(lambda v: f"{v:.1%}")
    cell["Company"] = cell.Company.astype(str).str[:34]
    if "Sector" in cell:
        cell["Sector"] = cell.Sector.astype(str).str[:24]
    tb = ax.table(cellText=cell.values, colLabels=[c.replace("_", " ") for c in cols], loc="upper left",
                  cellLoc="left", colLoc="left")
    tb.auto_set_font_size(False)
    tb.set_fontsize(7.5)
    tb.auto_set_column_width(list(range(len(cols))))
    tb.scale(1, 1.25)
    for (r, c), cl in tb.get_celld().items():
        cl.set_edgecolor(C.GRID)
        if r == 0:
            cl.set_text_props(weight="bold")
            cl.set_facecolor("#f0efec")
        elif cols[c] == "State":
            s = cell.iloc[r - 1, c]
            cl.set_facecolor(matplotlib.colors.to_rgba(C.STATE_COLORS.get(s, C.SURFACE), 0.25))
    ax.set_title("Risk dashboard — highest predicted 24-month IBC probability (latest fiscal year)", loc="left")
    return _save(fig, name)


def eq_forest(reg, name="F36_earnings_quality_by_definition.png"):
    if reg is None or reg.empty:
        return None
    outs = list(reg.Outcome.unique())
    fig, axes = plt.subplots(1, len(outs), figsize=(5.2 * len(outs), 0.42 * reg.Treatment.nunique() + 1.6),
                             squeeze=False)
    for ax, o in zip(axes[0], outs):
        t = reg[reg.Outcome == o].iloc[::-1]
        y = np.arange(len(t))
        ax.hlines(y, t.coef - 1.96 * t.se, t.coef + 1.96 * t.se, color=C.SERIES[0], lw=2)
        ax.scatter(t.coef, y, s=45, color=[C.SERIES[1] if p < 0.05 else C.SERIES[0] for p in t.p],
                   zorder=3, edgecolor=C.SURFACE, linewidth=1.2)
        ax.axvline(0, color=C.MUTED, ls="--", lw=1)
        ax.set_yticks(y, t.Treatment)
        ax.set_xlabel(f"effect of zombie status on {o.replace('_', ' ')} (95 % CI)")
        ax.grid(axis="y", visible=False)
        ax.set_title(o.replace("_", " "))
    fig.suptitle("Do zombies manage earnings? Panel OLS with sector & year FE, firm-clustered SE",
                 x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    return _save(fig, name, "Orange = significant at 5 %. μZ = fuzzy membership (dose–response).")


**`zew_analysis/synthetic.py`**

In [ ]:
%%writefile zew_analysis/synthetic.py
"""
Synthetic data — two uses only, never mixed with real results:

  healthy_universe(n)   V1 null-universe test: every firm ICR > 4, FCF > 0, RE > 0.
  make_demo_data(dir)   a realistic fake ABT in the pipeline's format, so the whole
                        analysis (and every chart) can be tried before the real scrape
                        finishes. All demo firm names start with "DEMO".
"""
import os

import numpy as np
import pandas as pd

SECTORS = ["Capital Goods", "Healthcare", "Automobile and Auto Components", "Fast Moving Consumer Goods",
           "Chemicals", "Information Technology", "Consumer Services", "Metals & Mining", "Power",
           "Construction", "Realty", "Textiles", "Oil Gas & Consumable Fuels", "Construction Materials"]
FYS = list(range(2015, 2027))


def _macro():
    return pd.DataFrame({"FY": FYS, "gsec10y": np.linspace(7.8, 6.9, len(FYS)),
                         "tbill3m": np.linspace(7.0, 6.5, len(FYS)),
                         "lending_rate": np.linspace(10.0, 9.0, len(FYS)),
                         "India_Total_ERP": np.linspace(0.085, 0.075, len(FYS))})


def _finish(rows):
    df = pd.DataFrame(rows)
    df = df.merge(_macro(), on="FY", how="left")
    df["FY_End"] = pd.to_datetime(df.FY.astype(str) + "-03-31")
    df["Filing_Date"] = df.FY_End + pd.DateOffset(months=2)
    return df


def _firm_year(rng, i, fy, TA, health, sector, shares, inc_year, list_year, price_idx, idx_level):
    debt_ratio = np.clip(0.25 - 0.18 * health + rng.normal(0, 0.04), 0.0, 0.85)
    TD = TA * debt_ratio
    st_share = np.clip(0.35 - 0.1 * health + rng.normal(0, 0.08), 0.05, 0.95)
    rate = np.clip(0.095 + rng.normal(0, 0.012) - (0.015 if health < -1.2 and rng.random() < 0.4 else 0), 0.04, 0.16)
    interest = TD * rate
    ebitda = TA * np.clip(0.11 + 0.07 * health + rng.normal(0, 0.015), -0.15, 0.4)
    dep = TA * 0.035
    ebit = ebitda - dep
    rev = TA * np.clip(0.95 + 0.15 * health + rng.normal(0, 0.1), 0.2, 3)
    cfo = ebitda * np.clip(0.8 + rng.normal(0, 0.15), -0.5, 1.4) - (0.03 * TA if health < -1 else 0)
    capex = TA * np.clip(0.05 + 0.02 * health + rng.normal(0, 0.015), 0.0, 0.2)
    pbt = ebit - interest
    ni = pbt * (0.75 if pbt > 0 else 1.0)
    cash = TA * np.clip(0.06 + 0.04 * health + rng.normal(0, 0.02), 0.002, 0.3)
    eq = TA * np.clip(0.45 + 0.15 * health, -0.3, 0.85)
    reserves = eq - TA * 0.03
    price = max(0.5, price_idx)
    mcap = price * shares / 1e7
    return {
        "ISIN_Root": f"DEMO{i:05d}", "Screener_Key": f"DEMO{i:04d}",
        "NSE_Symbol": f"DEMO{i:04d}", "Company": f"DEMO Company {i:04d} Ltd", "Industry": sector,
        "FY": fy, "Revenue": rev, "EBITDA": ebitda, "Depreciation": dep, "EBIT": ebit, "Interest": interest,
        "PBT": pbt, "Tax_Pct": 25.0 if pbt > 0 else 0.0, "Net_Profit": ni, "Profit_Excl_Exceptional": ni,
        "Other_Income": TA * 0.005, "CFO": cfo, "Capex": capex, "Net_Capex": capex * 0.95,
        "FCF": cfo - capex, "Total_Assets": TA, "Total_Debt": TD, "ST_Borrowings": TD * st_share,
        "LT_Borrowings": TD * (1 - st_share), "Cash_Equivalents": cash, "Net_Debt": TD - cash,
        "Reserves": reserves, "Equity_Capital": TA * 0.03, "Total_Equity": eq,
        "Trade_Receivables": rev * np.clip(0.18 - 0.04 * health + rng.normal(0, 0.03), 0.02, 0.6),
        "Inventories": rev * 0.12, "Current_Assets_approx": TA * 0.45,
        "Current_Liabilities_approx": TA * 0.25 + TD * st_share, "Net_Block": TA * 0.4,
        "Investments": TA * 0.05, "COGS": rev * np.clip(0.62 - 0.05 * health, 0.3, 0.95),
        "SGA_Other_Cost": rev * 0.12, "Shares_from_EPS": shares, "Shares_Best": shares,
        "Price_FY_End": price, "MCap_Cr": mcap, "NIFTY500_Level_FY_End": idx_level,
        "Equity_Vol_Ann": np.clip(0.3 - 0.06 * health + rng.normal(0, 0.05), 0.12, 1.2),
        "Inc_Year": inc_year, "NSE_Listing_Date": f"{list_year}-06-01",
        "Pledge_Pct_of_Promoter": (np.clip(20 - 12 * health + rng.normal(0, 8), 0, 95) if fy >= 2022 else np.nan),
        "Is_Financial": False,
    }


def healthy_universe(n=751, seed=2026):
    """Every firm comfortably solvent (ICR > 4, FCF > 0, RE > 0) in every year."""
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        TA = float(np.exp(rng.normal(7.5, 1.3)))
        sector = SECTORS[i % len(SECTORS)]
        for fy in FYS:
            r = _firm_year(rng, i, fy, TA, 1.2 + abs(rng.normal(0, 0.3)), sector, 1e8, 1990, 2005, 100, 10000)
            r["Interest"] = min(r["Interest"], r["EBIT"] / 4.5, r["CFO"] / 4.5)
            r["Interest"] = max(r["Interest"], 1e-3)
            r["Capex"] = min(r["Capex"], 0.6 * r["CFO"])
            r["FCF"] = r["CFO"] - r["Capex"]
            rows.append(r)
            TA *= 1.06
    df = _finish(rows)
    df["Sector"] = df.Industry
    return df


def make_demo_data(out_dir="zew_data_demo", n=600, seed=2026):
    """Write a synthetic 09_abt_firm_year.csv (+ price files) in the pipeline's format."""
    rng = np.random.default_rng(seed)
    rows = []
    idx = 10000 * np.cumprod(1 + rng.normal(0.10, 0.12, len(FYS)))
    for i in range(n):
        sector = SECTORS[rng.integers(len(SECTORS))]
        TA = float(np.exp(rng.normal(7.2, 1.4)))
        distressed = rng.random() < 0.18
        h = rng.normal(0.6, 0.6) if not distressed else rng.normal(0.0, 0.5)
        drift = -0.35 if distressed else 0.0
        inc = int(rng.integers(1950, 2016))
        lst = max(inc + 1, int(rng.integers(1990, 2018)))
        shares = float(np.exp(rng.normal(18.5, 1.0)))
        p = float(np.exp(rng.normal(4.5, 1.0)))
        dead = False
        for k, fy in enumerate(FYS):
            h = 0.75 * h + 0.25 * (0.6 if not distressed else -1.0) + drift * (k > 2) + rng.normal(0, 0.25)
            p *= np.exp(0.05 + 0.25 * (h - 0.3) * 0.5 + rng.normal(0, 0.25))
            r = _firm_year(rng, i, fy, TA, h, sector, shares, inc, lst, p, idx[k])
            r["Ret_1y"] = np.exp(0.05 + 0.125 * (h - 0.3)) - 1
            rows.append(r)
            TA *= np.exp(0.07 + 0.03 * h + rng.normal(0, 0.05))
            # hazard of IBC admission in the coming year
            if 2016 <= fy <= 2025:
                haz = 1 / (1 + np.exp(-(-6.0 - 2.2 * h)))
                if rng.random() < haz:
                    ev = min(pd.Timestamp(fy, 6, 1) + pd.Timedelta(days=int(rng.integers(30, 330))),
                             pd.Timestamp("2026-09-15"))
                    for rr in rows[-(k + 1):]:
                        rr["Event_Flag"] = 1
                        rr["Event_Flag_Substantive"] = int(rng.random() < 0.8)
                        rr["Event_Date"] = ev
                    dead = True
                    break
        if not dead:
            cens = pd.Timestamp("2026-09-30") if rng.random() > 0.04 else pd.Timestamp(2021, 9, 1)
            for rr in rows[-len(FYS):]:
                if rr.get("Screener_Key") == f"DEMO{i:04d}":
                    rr["Event_Flag"], rr["Event_Flag_Substantive"], rr["Censor_Date"] = 0, 0, cens
    df = _finish(rows)
    df = df[df.FY_End <= df.Event_Date.fillna(pd.Timestamp("2100-01-01")) + pd.DateOffset(months=12)]
    os.makedirs(out_dir, exist_ok=True)
    df.drop(columns=["FY_End", "Filing_Date"]).to_csv(os.path.join(out_dir, "09_abt_firm_year.csv"), index=False)
    print(f"demo ABT: {len(df):,} firm-years, {df.ISIN_Root.nunique()} firms, "
          f"{int(df.drop_duplicates('ISIN_Root').Event_Flag_Substantive.fillna(0).sum())} substantive IBC events "
          f"→ {out_dir}/09_abt_firm_year.csv")
    return out_dir


**`zew_analysis/report.py`**

In [ ]:
%%writefile zew_analysis/report.py
"""
Outputs:
  zew_results/ZEW_results.xlsx            every result table (one sheet each)
  zew_results/zew_firm_year_results.csv   the classified panel (label, state, scores, flags)
  zew_results/ZEW_report.html             self-contained report: headline numbers + all charts
"""
import base64
import html
import os

import numpy as np
import pandas as pd

from . import config as C

PANEL_COLS = ["ISIN_Root", "Company", "NSE_Symbol", "Sector", "FY", "Gate_Label", "State", "Trajectory",
              "ZEW", "P1", "P2", "P3", "P4", "P5", "Top_Pillar", "A", "B", "C", "D", "Veto",
              "G1_ICR_persist", "G2_CashICR_persist", "Incremental_CashICR", "Age_Unknown",
              "ICR", "CashICR", "Q", "FCF_3y", "EffRate", "ND_EBITDA", "RE_TA", "Merton_DD",
              *C.DEFINITIONS, "mu_Z", "Altman_Z", "Altman_Zone", "Beneish_M", "Beneish_Flag", "Piotroski_F",
              "Pledge_Pct", "CL_to_NetWorth", "Death", "Event_Date"]


def methodology():
    rows = [
        ("Layer 0", "Financial firms excluded", "spec 0.2", "Is_Financial / bank-format pages"),
        ("Layer 0", "Minimum usable years", "spec 0.3", C.MIN_USABLE_YEARS),
        ("Layer 0", "Sector bucket minimum", "spec 0.4", C.MIN_SECTOR_FIRMS),
        ("Layer 0", "Winsorisation (severity only)", "spec 0.5", C.WINSOR),
        ("Gate", "ICR / Cash ICR cut", "spec — economic break-even", C.ICR_CUT),
        ("Gate", "Persistence", "spec — AAM18 (2-y variant reported)", C.PERSIST_YEARS),
        ("Gate", "Firm age", "spec — AAM18 (5/10/15 grid)", C.AGE_MIN),
        ("Gate", "Age unknown policy", "impl", C.AGE_UNKNOWN_POLICY),
        ("Gate", "Tobin's Q cut", "spec — BH20 sector median", C.Q_PCTL),
        ("Gate", "Q statement fallback", "spec", "3-y revenue CAGR < 0 ∧ EBITDA-margin slope < 0"),
        ("Gate", "Effective-rate cut", "spec — CHK08 operationalised", C.EFF_RATE_PCTL),
        ("Gate", "Veto ICR / ND-EBITDA", "spec — rating-criteria anchor", f"{C.VETO_ICR} / {C.VETO_ND_EBITDA}"),
        ("Severity", "Pillar weights", "spec", C.PILLAR_WEIGHTS),
        ("Severity", "Override", "spec", C.OVERRIDE),
        ("Severity", "Within-pillar aggregation", "impl — non-compensatory", "max(mean, 0.75·max)"),
        ("Severity", "ND/EBITDA absolute anchor", "impl", 4.0),
        ("Severity", "Beta", "impl", "Blume-adjusted 2-y weekly vs Nifty 500; 1.0 if unavailable"),
        ("Severity", "Distance to default", "impl — Bharath & Shumway (2008) naive DD", "F = ST + 0.5·LT debt"),
        ("States", "Bands", "spec", f"{C.BAND_HEALTHY}/{C.BAND_WATCH}/{C.BAND_ATRISK}"),
        ("States", "Recovery", "spec", f"ICR > {C.RECOVERY_ICR} ∧ FCF > 0 for 2 y"),
        ("Benchmarks", "SST17 high leverage", "impl", f"sector-year p{int(C.SST_LEV_PCTL * 100)}"),
        ("Benchmarks", "ACEE20 rate benchmark", "impl", "AAA 5-y yield; fallback top-ICR-quintile median rate"),
        ("Benchmarks", "CHK08 rates", "impl", "prime = SBI MCLR/base rate → World Bank lending rate; LT = 10-y G-sec"),
        ("Benchmarks", "Beneish cut-off", "Beneish (1999)", f"{C.BENEISH_CUT} (−2.22 robustness)"),
        ("Fuzzy", "Ramps", "spec ICR; impl Q/FCF/CFO", "ICR 0.8–1.2; Q 0.8–1.2×median; FCF/CFO margin ±2 %"),
        ("Survival", "Death", "decision 28-Sep-2026", C.EVENT_COL),
        ("Survival", "Clock", "impl", "years since first classifiable filing date; zombie spell from zombie entry"),
        ("ML", "Label", "impl", f"IBC admission within {C.PD_HORIZON_MONTHS} months of filing date"),
        ("ML", "Split", "impl", f"train FY ≤ {C.TRAIN_LAST_FY}, test FY ≥ {C.TEST_FIRST_FY}"),
        ("Validation", "Bootstrap", "impl", f"{C.BOOTSTRAP_REPS} reps, ±{C.NOISE_SD:.0%} multiplicative noise"),
    ]
    return pd.DataFrame(rows, columns=["Layer", "Parameter", "Provenance", "Value"]).astype(str)


def _summary(R):
    d = R["panel"]
    comp = d[d.Gate_Computable]
    out = {
        "Firms analysed": d.ISIN_Root.nunique(),
        "Firm-years analysed": len(d),
        "Classifiable firm-years (3-y history)": len(comp),
        "Zombie firm-years (gate)": int(comp.Gate_Zombie.sum()),
        "  of which subsidised": int(comp.Gate_Subsidised.sum()),
        "Zombie prevalence (classifiable firm-years)": f"{comp.Gate_Zombie.mean():.1%}" if len(comp) else "–",
        "Firms ever zombie": int(d[d.Gate_Zombie].ISIN_Root.nunique()),
        "Incremental zombies via cash coverage only (G2)": int((comp.Gate_Zombie & comp.Incremental_CashICR).sum()),
        "IBC deaths (headline)": int(d.drop_duplicates("ISIN_Root").Death.sum()),
        "V1 null universe — ZEW zombies": int(R["v1"].Flagged.iat[0]) if len(R["v1"]) else "–",
        "V1 null universe — v2 percentile flags": int(R["v1"].Flagged.iat[1]) if len(R["v1"]) else "–",
        "V2 known-good false-positive rate": f"{R['v2'].get('zew_false_positive_rate', np.nan):.0%}" if R.get("v2") else "list not in sample",
        "V3 recall (zombie before IBC)": f"{R['v3'].get('recall_zombie', np.nan):.0%}" if R.get("v3") else "–",
        "V3 recall (At Risk or worse)": f"{R['v3'].get('recall_atrisk_plus', np.nan):.0%}" if R.get("v3") else "–",
        "V4 sector χ² p-value": f"{R['v4'].get('p_value', np.nan):.3g}" if R.get("v4") else "–",
        "V5 all-7 ÷ any-definition": f"{R['v5'].get('all_over_any', np.nan):.1%}" if R.get("v5") else "–",
    }
    if R.get("logrank"):
        out["Log-rank p (zombie vs non-zombie)"] = f"{R['logrank']['logrank_p']:.3g}"
    if isinstance(R.get("cox_main"), pd.DataFrame) and "Is_Zombie_State" in R["cox_main"].index:
        r = R["cox_main"].loc["Is_Zombie_State"]
        out["Cox HR zombie (time-varying)"] = f"{r.HR:.2f} [{r.HR_low:.2f}–{r.HR_high:.2f}]"
    m = R["ml"].get("metrics")
    if isinstance(m, pd.DataFrame) and "ROC_AUC" in m:
        for mod in ("PD_XGB", "PD_Logit"):
            r = m[m.Model == mod]
            if len(r):
                out[f"Test ROC-AUC {mod}"] = f"{r.ROC_AUC.iat[0]:.3f}"
    return pd.DataFrame(list(out.items()), columns=["Measure", "Value"])


def write(R):
    os.makedirs(C.RESULTS_DIR, exist_ok=True)
    d = R["panel"]
    d[[c for c in PANEL_COLS if c in d]].to_csv(os.path.join(C.RESULTS_DIR, "zew_firm_year_results.csv"), index=False)
    summ = _summary(R)
    sheets = {
        "Summary": summ, "Methodology": methodology(),
        "Funnel": pd.DataFrame(R["funnel"], columns=["Step", "Firm_years", "Firms"]) if R["funnel"] else None,
        "V1_null_universe": R.get("v1"), "V2_known_good": R.get("v2_table"),
        "V3_known_distressed": R.get("v3_table"), "V4_sector_balance": R.get("v4_table"),
        "V5_jaccard": R.get("jaccard"), "V5_prevalence": R.get("prevalence"),
        "V6_threshold_grid": R.get("v6_grid"),
        "V6_flip_prone": R["v6_boot"].sort_values("Flip_Rate", ascending=False).head(200) if len(R.get("v6_boot", [])) else None,
        "State_transitions": R.get("transition"),
        "Death_clock": R.get("death_clock"), "Cox_main": R.get("cox_main"), "Cox_by_definition": R.get("cox_defs"),
        "Shumway_hazard": R.get("shumway"), "ML_metrics": R["ml"].get("metrics"),
        "Label_metrics": R.get("label_metrics"),
        "SHAP_importance": R["ml"].get("shap_importance").to_frame() if R["ml"].get("shap_importance") is not None else None,
        "EQ_by_state": R.get("eq_profile"), "EQ_tests": R.get("eq_tests"),
        "EQ_regressions": R.get("eq_regressions"), "Incremental_G2": R.get("incremental_table"),
        "EventTime_accruals": R.get("et_accruals"), "EventTime_beneish": R.get("et_beneish"),
        "Risk_dashboard": R.get("risk_table"),
    }
    xl = os.path.join(C.RESULTS_DIR, "ZEW_results.xlsx")
    with pd.ExcelWriter(xl, engine="openpyxl") as w:
        for name, t in sheets.items():
            if isinstance(t, pd.Series):
                t = t.to_frame()
            if isinstance(t, pd.DataFrame) and len(t):
                tt = t.copy()
                for c in tt.columns:
                    if pd.api.types.is_datetime64_any_dtype(tt[c]):
                        tt[c] = tt[c].dt.date
                tt.to_excel(w, sheet_name=name[:31], index=not isinstance(tt.index, pd.RangeIndex))
    _html(R, summ)
    print(f"  tables → {xl}")


def _img(p):
    with open(p, "rb") as f:
        return "data:image/png;base64," + base64.b64encode(f.read()).decode()


def _table(df, n=25):
    if not isinstance(df, pd.DataFrame) or df.empty:
        return "<p class='muted'>not available</p>"
    return df.head(n).to_html(classes="t", float_format=lambda v: f"{v:,.3g}", border=0, na_rep="–")


def _html(R, summ):
    figs = R.get("figures", {})
    parts = [f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1"><title>ZEW 4.0 Results</title>
<style>
:root{{--bg:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--muted:#8a8984;--line:#e6e5e1;--accent:#2a78d6}}
@media (prefers-color-scheme: dark){{:root{{--bg:#1a1a19;--ink:#f0efec;--ink2:#c3c2b7;--muted:#97968f;--line:#383835;--accent:#3987e5}}
img{{background:#fcfcfb;border-radius:6px}}}}
body{{margin:0;background:var(--bg);color:var(--ink);font:15px/1.55 system-ui,-apple-system,"Segoe UI",sans-serif}}
main{{max-width:1080px;margin:0 auto;padding:24px 16px 64px}}
h1{{font-size:26px;margin:0 0 4px}} h2{{font-size:19px;margin:36px 0 10px;padding-top:12px;border-top:1px solid var(--line)}}
.muted{{color:var(--muted)}} .kpis{{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:10px;margin:18px 0}}
.kpi{{border:1px solid var(--line);border-radius:8px;padding:10px 12px}} .kpi b{{display:block;font-size:20px}}
.kpi span{{color:var(--ink2);font-size:13px}} img{{max-width:100%;height:auto;margin:10px 0 18px;display:block}}
table.t{{border-collapse:collapse;font-size:12.5px;margin:8px 0 18px;display:block;overflow-x:auto}}
table.t th,table.t td{{padding:4px 8px;border-bottom:1px solid var(--line);text-align:left;white-space:nowrap}}
nav a{{color:var(--accent);margin-right:12px;text-decoration:none}}
</style></head><body><main>
<h1>ZEW 4.0 — Zombie Early Warning: results</h1>
<p class="muted">Nifty 500 + Nifty Microcap 250 · FY{C.FY_MIN}–FY{C.FY_MAX} · death = IBC admission ({html.escape(C.EVENT_COL)})</p>
<nav>""" + "".join(f'<a href="#{html.escape(s)}">{html.escape(s)}</a>' for s in ["Summary", *figs.keys(), "Tables"]) + "</nav>"]
    parts.append('<h2 id="Summary">Headline numbers</h2><div class="kpis">' + "".join(
        f'<div class="kpi"><b>{html.escape(str(v))}</b><span>{html.escape(str(k))}</span></div>'
        for k, v in summ.values) + "</div>")
    for sec, ps in figs.items():
        parts.append(f'<h2 id="{html.escape(sec)}">{html.escape(sec)}</h2>')
        for p in ps:
            parts.append(f'<img src="{_img(p)}" alt="{html.escape(os.path.basename(p))}">')
    parts.append('<h2 id="Tables">Key tables</h2>')
    for title, t in (("Death clock (Kaplan–Meier)", R.get("death_clock")), ("Cox — main model", R.get("cox_main")),
                     ("Cox — by definition", R.get("cox_defs")), ("Discrete-time hazard (Shumway)", R.get("shumway")),
                     ("ML — test-set metrics", R["ml"].get("metrics")), ("Every flag / score vs IBC label", R.get("label_metrics")),
                     ("Earnings quality by state", R.get("eq_profile")), ("Earnings-quality regressions", R.get("eq_regressions")),
                     ("G2 incremental set", R.get("incremental_table")), ("Risk dashboard (top 25)", R.get("risk_table")),
                     ("Methodology & threshold provenance", methodology())):
        parts.append(f"<h3>{html.escape(title)}</h3>" + _table(t if not isinstance(t, pd.Series) else t.to_frame(), 40))
    parts.append("</main></body></html>")
    with open(os.path.join(C.RESULTS_DIR, "ZEW_report.html"), "w", encoding="utf-8") as f:
        f.write("\n".join(parts))


**`zew_analysis/run.py`**

In [ ]:
%%writefile zew_analysis/run.py
"""
End-to-end analysis: Layer 0 → Gate → Severity → States → Benchmarks → Fuzzy → Validation
→ Survival → ML → Forensic → charts → Excel + HTML report.

    from zew_analysis.run import run_all
    R = run_all(data_dir="zew_data", results_dir="zew_results")
"""
import os
import time
import warnings

import numpy as np
import pandas as pd

from . import benchmarks, config as C, data, forensic, fuzzy, gate, ml, plots, ratios, severity
from . import states, survival, validation


def _log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def build(data_dir=None, use_prices=True):
    """Layers 0–4 + fuzzy. Returns the enriched firm-year panel."""
    if data_dir:
        C.DATA_DIR = data_dir
    _log("Layer 0 — loading panel")
    raw = data.load_panel()
    px, ix = data.load_prices() if use_prices else (None, None)
    _log("ratios")
    d = ratios.compute(raw, px, ix)
    _log("Layer 1 — gate")
    d = gate.apply(d)
    _log("Layer 2 — severity")
    d = severity.apply(d)
    _log("Layer 3 — states & trajectory")
    d = states.apply(d)
    d["AtRisk_or_Worse"] = d.State.isin(["At Risk", "Zombie", "Subsidised Zombie"])
    _log("Layer 4 — benchmarks (6 definitions, Altman, Beneish, Piotroski)")
    d = benchmarks.apply(d)
    _log("fuzzy variant")
    d = fuzzy.apply(d)
    d["v2_Flag"] = validation.v2_percentile_flag(d)
    d = d.loc[:, ~d.columns.duplicated()]
    return raw, d


def run_all(data_dir=None, results_dir=None, bootstrap_reps=None, make_charts=True, use_prices=True):
    warnings.filterwarnings("ignore")
    if results_dir:
        C.RESULTS_DIR = results_dir
        C.FIG_DIR = os.path.join(results_dir, "figures")
    os.makedirs(C.FIG_DIR, exist_ok=True)
    plots.style()
    t0 = time.time()
    raw, d = build(data_dir, use_prices)
    R = {"panel": d, "funnel": raw.attrs.get("funnel", [])}

    _log("validation V1–V6")
    R["v1"] = validation.v1_null_universe()
    R["v2_table"], R["v2"] = validation.v2_known_good(d)
    R["v3_table"], R["v3"] = validation.v3_known_distressed(d)
    R["v4_table"], R["v4"] = validation.v4_sector_balance(d)
    R["jaccard"], R["prevalence"], R["v5"] = validation.v5_jaccard(d)
    R["v6_grid"] = validation.v6_grids(d)
    reps = C.BOOTSTRAP_REPS if bootstrap_reps is None else bootstrap_reps
    R["v6_boot"] = validation.v6_bootstrap(raw, d, reps=reps) if reps else pd.DataFrame()
    R["transition"] = states.transition_matrix(d)

    _log("survival — Kaplan–Meier, log-rank, Cox (time-varying), Shumway hazard")
    sp = survival.spells(d)
    R["spells"] = sp
    if len(sp) and sp.Event.sum() > 0:
        R["km_fits"], R["logrank"] = survival.km_logrank(sp)
        R["death_clock"] = survival.death_clock(R["km_fits"])
        km_defs = {}
        for dfn in C.DEFINITIONS:
            s2 = survival.spells(d, flag=dfn)
            if len(s2) and (s2.Group == "Zombie").any():
                km_defs[dfn] = survival.km_logrank(s2)
        R["km_defs"] = km_defs
        cp = survival.counting_process(d)
        R["cox_model"], R["cox_main"] = survival.cox_tv(cp, ["Is_Zombie_State", "Subsidised_State", "ZEW10",
                                                             "log_TA", "Lev"])
        R["cox_defs"] = survival.cox_by_definition(d)
        R["shumway_model"], R["shumway"] = survival.shumway_logit(d)
        R["cox_prof_model"], R["cox_profiles"] = survival.cox_profiles(sp)
    else:
        R["km_fits"], R["logrank"], R["km_defs"] = {}, {}, {}
        _log("  !! no events in usable firms — survival layer skipped")

    _log("machine learning — 24-month PD (logit, XGBoost, XGBoost+SMOTE)")
    panel, feats = ml.build_panel(d)
    R["ml_panel_n"] = len(panel)
    res = ml.fit_all(panel, feats)
    R["ml"] = res
    R["risk_table"] = ml.score_latest(d, res, feats) if "models" in res else pd.DataFrame()
    if "error" in res:
        _log(f"  !! {res['error']}")
    scores = {k: panel[c] * s for k, c, s in (("ZEW severity", "ZEW", 1), ("Fuzzy μZ", "mu_Z", 1),
                                               ("Altman Z″ (−)", "Altman_Z", -1), ("Merton DD (−)", "Merton_DD", -1),
                                               ("Beneish M", "Beneish_M", 1)) if c in panel}
    for dfn in C.DEFINITIONS:
        scores[f"flag: {dfn}"] = panel[dfn].astype(float)
    R["label_metrics"] = validation.label_metrics(panel, scores)

    _log("forensic — earnings quality vs zombie status")
    R["eq_profile"], R["eq_tests"] = forensic.profile_by_state(d)
    R["eq_regressions"] = forensic.regressions(d)
    R["et_accruals"] = forensic.event_time(d, "Sloan_Accruals")
    R["et_beneish"] = forensic.event_time(d, "Beneish_M")
    R["incremental_table"], inc_x = forensic.incremental_set(d)

    if make_charts:
        _log("charts")
        R["figures"] = charts(R, inc_x)
    from . import report
    report.write(R)
    _log(f"done in {(time.time() - t0) / 60:.1f} min → {C.RESULTS_DIR}/")
    return R


def charts(R, inc_x=None):
    d = R["panel"]
    figs = {}
    jobs = [
        ("Data", "F01", lambda: plots.funnel(R["funnel"])),
        ("Data", "F02", lambda: plots.events_by_year(d)),
        ("Data", "F03", lambda: plots.coverage_heatmap(d)),
        ("Data", "F04", lambda: plots.coverage_distributions(d)),
        ("Layer 1 — Gate", "F05", lambda: plots.prevalence_by_year(d)),
        ("Layer 1 — Gate", "F06", lambda: plots.gate_combinations(d)),
        ("Layer 1 — Gate", "F07", lambda: plots.icr_vs_cashicr(d)),
        ("Layer 1 — Gate", "F08", lambda: plots.sector_prevalence(R["v4_table"], R["v4"])),
        ("Layer 2 — Severity", "F09", lambda: plots.severity_by_state(d)),
        ("Layer 2 — Severity", "F10", lambda: plots.pillars_by_state(d)),
        ("Layer 3 — States", "F11", lambda: plots.trajectories(d)),
        ("Layer 3 — States", "F12", lambda: plots.transition_heatmap(R["transition"])),
        ("Layer 3 — States", "F13", lambda: plots.state_share(d)),
        ("Layer 4 — Benchmarks", "F14", lambda: plots.definitions_by_year(d)),
        ("Layer 4 — Benchmarks", "F15", lambda: plots.jaccard_heatmap(R["jaccard"], R["v5"])),
        ("Layer 4 — Benchmarks", "F16", lambda: plots.altman_by_state(d)),
        ("Validation", "F17", lambda: plots.v1_null(R["v1"])),
        ("Validation", "F18", lambda: plots.v3_lead(R["v3_table"])),
        ("Validation", "F19", lambda: plots.v6_grid(R["v6_grid"])),
        ("Validation", "F20", lambda: plots.flip_rates(R["v6_boot"])),
        ("Validation", "F21", lambda: plots.fuzzy_distribution(d)),
        ("Survival", "F22", lambda: plots.km_plot(R.get("km_fits"), R.get("logrank"))),
        ("Survival", "F23", lambda: plots.km_by_definition(R.get("km_defs"))),
        ("Survival", "F24", lambda: plots.forest(R.get("cox_defs"))),
        ("Survival", "F25", lambda: plots.cox_profile_curves(R.get("cox_profiles"))),
        ("Machine learning", "F26", lambda: plots.roc_pr(R["ml"].get("preds"))),
        ("Machine learning", "F27", lambda: plots.calibration(R["ml"].get("preds"))),
        ("Machine learning", "F28", lambda: plots.shap_summary(R["ml"])),
        ("Machine learning", "F29", lambda: plots.confusion(R["ml"].get("preds"))),
        ("Machine learning", "F30", lambda: plots.pd_distribution(R["ml"].get("preds"))),
        ("Forensic", "F31", lambda: plots.beneish_by_state(d)),
        ("Forensic", "F32", lambda: plots.event_time_plot(R["et_accruals"], R["et_beneish"])),
        ("Forensic", "F33", lambda: plots.incremental_plot(inc_x)),
        ("Forensic", "F34", lambda: plots.pledge_by_state(d)),
        ("Forensic", "F36", lambda: plots.eq_forest(R["eq_regressions"])),
        ("Dashboard", "F35", lambda: plots.risk_table_fig(R["risk_table"])),
    ]
    for sec, key, fn in jobs:
        try:
            p = fn()
        except Exception as e:                                       # noqa: BLE001
            _log(f"  chart {key} skipped: {type(e).__name__}: {e}")
            p = None
        if p:
            figs.setdefault(sec, []).append(p)
    n = sum(len(v) for v in figs.values())
    _log(f"  {n} charts → {C.FIG_DIR}")
    return figs


**`zew_analysis/__main__.py`**

In [ ]:
%%writefile zew_analysis/__main__.py
"""
python -m zew_analysis                       # real data: zew_data/ → zew_results/
python -m zew_analysis --demo                # synthetic demo data (try everything first)
python -m zew_analysis --data zew_data --out zew_results --boot 200
"""
import argparse
import sys


def main(argv=None):
    ap = argparse.ArgumentParser(prog="zew_analysis")
    ap.add_argument("--data", default="zew_data", help="pipeline output folder")
    ap.add_argument("--out", default="zew_results", help="results folder")
    ap.add_argument("--boot", type=int, default=None, help="bootstrap reps for V6 (0 = skip)")
    ap.add_argument("--demo", action="store_true", help="generate and analyse synthetic demo data")
    ap.add_argument("--no-prices", action="store_true", help="skip weekly-price beta estimation")
    a = ap.parse_args(argv)
    from .run import run_all
    data = a.data
    if a.demo:
        from .synthetic import make_demo_data
        data = make_demo_data("zew_data_demo")
        if a.out == "zew_results":
            a.out = "zew_results_demo"
    run_all(data_dir=data, results_dir=a.out, bootstrap_reps=a.boot, use_prices=not a.no_prices)


if __name__ == "__main__":
    sys.exit(main())


**`tests/test_analysis.py`** — self-tests (null universe, gate rules, age filter, Cash-ICR extension, anchoring, end-to-end)

In [ ]:
%%writefile tests/test_analysis.py
"""
Offline tests for the analysis layer. Run:  python tests/test_analysis.py
"""
import os
import sys
import tempfile

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from zew_analysis import config as C, gate, ratios, severity, synthetic, validation   # noqa: E402


def _firm(icr_path, cfo_ratio=1.0, q=0.6, age=30, n=6, debt=100.0):
    rows = []
    for k, icr in enumerate(icr_path[:n]):
        interest = 10.0
        ebit = icr * interest
        rows.append({"ISIN_Root": "X", "FY": 2015 + k, "Industry": "Test", "Sector": "Test",
                     "EBIT": ebit, "EBITDA": ebit + 5, "Interest": interest, "CFO": cfo_ratio * ebit,
                     "Capex": 8.0, "FCF": cfo_ratio * ebit - 8.0, "Revenue": 200.0, "Total_Assets": 300.0,
                     "Total_Debt": debt, "Net_Debt": debt - 5, "Cash_Equivalents": 5.0, "Reserves": 50.0,
                     "Total_Equity": 120.0, "MCap_Cr": q * 300 - debt if q * 300 > debt else 10.0,
                     "Inc_Year": 2015 + k - age, "FY_End": pd.Timestamp(2015 + k, 3, 31)})
    return pd.DataFrame(rows)


def test_null_universe_zero():
    v1 = validation.v1_null_universe(n=120)
    assert v1.Flagged.iat[0] == 0, "ZEW gate must flag nobody in a healthy universe"
    assert 0.25 < v1.Flagged_Pct.iat[1] / 100 < 0.40, "percentile rule should flag about one third"


def test_gate_zombie_and_veto():
    # chronic ICR < 1 with negative cash flow → zombie in year 3+
    peers = pd.concat([_firm([5] * 6).assign(ISIN_Root=f"P{i}", MCap_Cr=400 + i) for i in range(6)])
    z = _firm([0.5, 0.6, 0.4, 0.3, 0.5, 0.2], cfo_ratio=-0.5)
    df = pd.concat([peers, z], ignore_index=True)
    d = ratios.compute(df)
    g = gate.run_gate(d)
    zz = g[d.ISIN_Root == "X"]
    assert zz.Gate_Zombie.iloc[2:].all() and not zz.Gate_Zombie.iloc[:2].any()
    # healthy veto: ICR 5, ND/EBITDA < 2, FCF > 0, RE > 0
    assert g[d.ISIN_Root == "P0"].Veto.iloc[2:].all()


def test_age_filter():
    young = _firm([0.5] * 6, cfo_ratio=-0.5, age=3)
    d = ratios.compute(young)
    assert not gate.run_gate(d).Gate_Zombie.any(), "a 3-year-old firm is a start-up, not a zombie"
    assert gate.run_gate(d, age_min=0).Gate_Zombie.iloc[2:].all()


def test_cash_icr_extension():
    # EBIT says fine (ICR 1.5) but cash says not (CFO ≈ 0.3 × EBIT → CashICR 0.45)
    peers = pd.concat([_firm([5] * 6).assign(ISIN_Root=f"P{i}", MCap_Cr=400 + i) for i in range(6)])
    x = _firm([1.5] * 6, cfo_ratio=0.3)
    d = ratios.compute(pd.concat([peers, x], ignore_index=True))
    g = gate.run_gate(d)
    assert g[d.ISIN_Root == "X"].A.iloc[2:].all(), "G2 must catch accrual-only coverage"
    assert not gate.run_gate(d, use_cash_icr=False)[d.ISIN_Root == "X"].A.any()


def test_severity_anchored():
    # if every firm in a sector has ICR ≈ 12, nobody is pushed into the tail
    rows = [_firm([12 + 0.01 * i] * 6).assign(ISIN_Root=f"F{i}") for i in range(20)]
    d = ratios.compute(pd.concat(rows, ignore_index=True))
    s = severity.score_variables(d)
    assert s.S_ICR.max() < 50, "a sector of uniformly healthy coverage must not score as distressed"


def test_demo_end_to_end():
    from zew_analysis.run import run_all
    with tempfile.TemporaryDirectory() as tmp:
        data = synthetic.make_demo_data(os.path.join(tmp, "data"), n=160)
        R = run_all(data_dir=data, results_dir=os.path.join(tmp, "res"), bootstrap_reps=3,
                    make_charts=False, use_prices=False)
        assert os.path.exists(os.path.join(tmp, "res", "ZEW_results.xlsx"))
        assert R["panel"].Gate_Zombie.sum() > 0 and R["v1"].Flagged.iat[0] == 0


if __name__ == "__main__":
    fails = 0
    for name, fn in sorted(globals().items()):
        if name.startswith("test_") and callable(fn):
            try:
                fn()
                print(f"PASS  {name}")
            except Exception as e:                                   # noqa: BLE001
                fails += 1
                print(f"FAIL  {name}: {type(e).__name__}: {e}")
    sys.exit(1 if fails else 0)


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "tests/test_analysis.py"], capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])
assert r.returncode == 0, "A self-test failed — send me the output above."

## Part 2 — Settings
* `USE_DEMO = False` → analyse the real pipeline output in `DATA_DIR` (default `zew_data`).
* `USE_DEMO = True`  → generate synthetic data and analyse it (for practice; never report these numbers).
* `BOOT_REPS` — V6 bootstrap replications (200 for the thesis; 20 while experimenting).

In [ ]:
USE_DEMO    = False
DATA_DIR    = "zew_data"          # folder written by the pipeline notebook
RESULTS_DIR = "zew_results"
BOOT_REPS   = 50

import importlib, os, sys, warnings
import pandas as pd, numpy as np
from IPython.display import Image, display
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40, "display.width", 200)
sys.path.insert(0, os.getcwd())
import zew_analysis
from zew_analysis import (config as C, data, ratios, gate, severity, states, benchmarks, fuzzy,
                          validation, survival, ml, forensic, plots, synthetic, report)
for m in (C, data, ratios, gate, severity, states, benchmarks, fuzzy, validation, survival, ml, forensic, plots, synthetic, report):
    importlib.reload(m)

if USE_DEMO:
    DATA_DIR = synthetic.make_demo_data("zew_data_demo")
    RESULTS_DIR = "zew_results_demo"
C.DATA_DIR, C.RESULTS_DIR = DATA_DIR, RESULTS_DIR
C.FIG_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(C.FIG_DIR, exist_ok=True)
plots.style()
R = {}

def show(path):
    if path: display(Image(filename=path))
    else: print("(chart skipped — not enough data for it yet)")
print("data:", DATA_DIR, "| results:", RESULTS_DIR)

## Part 3 — Layer 0: analysis panel
Financial firms out (0.2), ≥ 5 usable years (0.3), sector buckets (0.4), filing date = FY end + 2 months.

In [ ]:
raw = data.load_panel()
px, ix = data.load_prices()
R["funnel"] = raw.attrs.get("funnel", [])
d = ratios.compute(raw, px, ix)
print(f"{len(d):,} firm-years · {d.ISIN_Root.nunique():,} firms · FY{d.FY.min()}–FY{d.FY.max()}")
show(plots.funnel(R["funnel"])); show(plots.events_by_year(d))

In [ ]:
show(plots.coverage_heatmap(d)); show(plots.coverage_distributions(d))

## Part 4 — Layer 1: the Gate
`Zombie = A ∧ (B ∨ C)` · `Subsidised = Zombie ∧ D` · veto first. Also computed: the 2-year (BH20-style) and no-Cash-ICR variants.

In [ ]:
d = gate.apply(d)
comp = d[d.Gate_Computable]
display(comp.Gate_Label.value_counts().to_frame("firm-years"))
print("Zombie prevalence:", f"{comp.Gate_Zombie.mean():.1%}",
      "| 2-year variant:", f"{comp.Gate_Zombie_2y.mean():.1%}",
      "| without Cash ICR (pure AAM18 coverage):", f"{comp.Gate_Zombie_noCashICR.mean():.1%}",
      "| extra zombies caught only by Cash ICR (G2):", int((comp.Gate_Zombie & comp.Incremental_CashICR).sum()))
show(plots.prevalence_by_year(d)); show(plots.gate_combinations(d)); show(plots.icr_vs_cashicr(d))

## Part 5 — Layer 2: severity and root cause
Sector-year robust z-scores → 0–100, five pillars, non-compensatory override. Diagnostic only.

In [ ]:
d = severity.apply(d)
display(d[["ZEW", "P1", "P2", "P3", "P4", "P5"]].describe().round(1))

## Part 6 — Layer 3: states and trajectories

In [ ]:
d = states.apply(d)
d["AtRisk_or_Worse"] = d.State.isin(["At Risk", "Zombie", "Subsidised Zombie"])
R["transition"] = states.transition_matrix(d)
display(d.State.value_counts().to_frame("firm-years"))
show(plots.severity_by_state(d)); show(plots.pillars_by_state(d))
show(plots.state_share(d)); show(plots.transition_heatmap(R["transition"])); show(plots.trajectories(d))

## Part 7 — Layer 4: benchmarks
Six published zombie definitions run in parallel, plus Altman Z″, Beneish M and Piotroski F.

In [ ]:
d = benchmarks.apply(d)
d = fuzzy.apply(d)                                   # Part 8 (fuzzy) computed here too
d["v2_Flag"] = validation.v2_percentile_flag(d)
d = d.loc[:, ~d.columns.duplicated()]
R["panel"] = d
display((d[d.Gate_Computable][C.DEFINITIONS].mean() * 100).round(2).to_frame("prevalence %"))
show(plots.definitions_by_year(d)); show(plots.altman_by_state(d))

## Part 8 — Fuzzy variant
Continuous membership μZ ∈ [0, 1] — robustness specification and continuous treatment.

In [ ]:
show(plots.fuzzy_distribution(d))

## Part 9 — Validation (V1–V6)

In [ ]:
R["v1"] = validation.v1_null_universe()
display(R["v1"]); show(plots.v1_null(R["v1"]))
R["v2_table"], R["v2"] = validation.v2_known_good(d)
print("V2 known-good list:", R["v2"]); display(R["v2_table"])

In [ ]:
R["v3_table"], R["v3"] = validation.v3_known_distressed(d)
print("V3:", R["v3"]); display(R["v3_table"].head(20)); show(plots.v3_lead(R["v3_table"]))
R["v4_table"], R["v4"] = validation.v4_sector_balance(d)
print("V4:", R["v4"]); show(plots.sector_prevalence(R["v4_table"], R["v4"]))

In [ ]:
R["jaccard"], R["prevalence"], R["v5"] = validation.v5_jaccard(d)
print("V5:", R["v5"]); show(plots.jaccard_heatmap(R["jaccard"], R["v5"]))
R["v6_grid"] = validation.v6_grids(d); show(plots.v6_grid(R["v6_grid"]))
R["v6_boot"] = validation.v6_bootstrap(raw, d, reps=BOOT_REPS) if BOOT_REPS else pd.DataFrame()
show(plots.flip_rates(R["v6_boot"]))

## Part 10 — Survival analysis: the death clock
Zombie vs non-zombie spells (no immortal-time bias), log-rank, Cox with time-varying covariates, Shumway discrete-time hazard.

In [ ]:
sp = survival.spells(d); R["spells"] = sp
print(sp.groupby("Group").agg(spells=("T", "size"), events=("Event", "sum"), mean_years=("T", "mean")))
if len(sp) and sp.Event.sum() > 0:
    R["km_fits"], R["logrank"] = survival.km_logrank(sp)
    R["death_clock"] = survival.death_clock(R["km_fits"])
    print("log-rank:", R["logrank"]); display(R["death_clock"])
    show(plots.km_plot(R["km_fits"], R["logrank"]))
else:
    R["km_fits"], R["logrank"] = {}, {}
    print("No IBC events among usable firms yet — survival analysis needs the full events run.")

In [ ]:
R["km_defs"] = {}
for dfn in C.DEFINITIONS:
    s2 = survival.spells(d, flag=dfn)
    if len(s2) and (s2.Group == "Zombie").any() and s2.Event.sum() > 0:
        R["km_defs"][dfn] = survival.km_logrank(s2)
show(plots.km_by_definition(R["km_defs"]))
cp = survival.counting_process(d)
R["cox_model"], R["cox_main"] = survival.cox_tv(cp, ["Is_Zombie_State", "Subsidised_State", "ZEW10", "log_TA", "Lev"])
print("Cox (time-varying) — main model"); display(R["cox_main"])
R["cox_defs"] = survival.cox_by_definition(d); display(R["cox_defs"]); show(plots.forest(R["cox_defs"]))
R["shumway_model"], R["shumway"] = survival.shumway_logit(d)
print("Shumway discrete-time hazard (odds ratios)"); display(R["shumway"])
R["cox_prof_model"], R["cox_profiles"] = survival.cox_profiles(sp) if len(sp) else (None, None)
show(plots.cox_profile_curves(R["cox_profiles"]))

## Part 11 — Machine learning: 24-month probability of default
Out-of-time split (train FY ≤ 2021, test FY ≥ 2022). Logit benchmark, XGBoost with `scale_pos_weight`, XGBoost + SMOTE; TreeSHAP explanations.

In [ ]:
panel, feats = ml.build_panel(d)
res = ml.fit_all(panel, feats); R["ml"] = res
print({k: res[k] for k in ("train_n", "test_n", "train_events", "test_events")}, res.get("error", ""))
if "metrics" in res:
    display(res["metrics"].round(3))
    show(plots.roc_pr(res["preds"])); show(plots.calibration(res["preds"]))
    show(plots.shap_summary(res)); show(plots.confusion(res["preds"])); show(plots.pd_distribution(res["preds"]))
scores = {k: panel[c] * s for k, c, s in (("ZEW severity", "ZEW", 1), ("Fuzzy μZ", "mu_Z", 1),
          ("Altman Z″ (−)", "Altman_Z", -1), ("Merton DD (−)", "Merton_DD", -1), ("Beneish M", "Beneish_M", 1)) if c in panel}
for dfn in C.DEFINITIONS:
    scores[f"flag: {dfn}"] = panel[dfn].astype(float)
R["label_metrics"] = validation.label_metrics(panel, scores)
print("Every definition and score against the IBC label"); display(R["label_metrics"])

## Part 12 — Forensic: do zombies manipulate earnings?

In [ ]:
R["eq_profile"], R["eq_tests"] = forensic.profile_by_state(d)
display(R["eq_profile"].round(3)); display(R["eq_tests"])
R["eq_regressions"] = forensic.regressions(d); display(R["eq_regressions"])
show(plots.eq_forest(R["eq_regressions"])); show(plots.beneish_by_state(d))
R["et_accruals"], R["et_beneish"] = forensic.event_time(d, "Sloan_Accruals"), forensic.event_time(d, "Beneish_M")
show(plots.event_time_plot(R["et_accruals"], R["et_beneish"]))
R["incremental_table"], inc_x = forensic.incremental_set(d); display(R["incremental_table"])
show(plots.incremental_plot(inc_x)); show(plots.pledge_by_state(d))

## Part 13 — Risk dashboard and report
Latest-year PD for every live firm, then the Excel workbook, firm-year CSV and HTML report with every chart.

In [ ]:
R["risk_table"] = ml.score_latest(d, res, feats) if "models" in res else pd.DataFrame()
display(R["risk_table"].head(25)); show(plots.risk_table_fig(R["risk_table"]))
from zew_analysis.run import charts
R["figures"] = charts(R, inc_x)
report.write(R)
print("Open", os.path.join(RESULTS_DIR, "ZEW_report.html"), "in a browser; tables in", os.path.join(RESULTS_DIR, "ZEW_results.xlsx"))